# Eksperimen lanjutan v3 — PAPER024914 (Google Colab + Google Drive)

Notebook v3 ini adalah **lanjutan v2**: isinya sama dengan v2 ditambah **Tahap 1b** (SMOTE tanpa class weight). Semua keluaran tetap ditulis ke folder **`results_v2/`** yang sama, sehingga run Tahap 1 yang sudah selesai (5 seed, dijalankan dengan v2) **dilewati otomatis** dan tidak dilatih ulang.

Protokolnya sama persis dengan mode `full` di `results/`: praproses yang sama, split 80/20 `random_state=42`, MinMax hanya di-fit pada train, test tidak diubah, CNN-LSTM yang sama, 50 epoch, early stopping patience 5, batch 512, float32. Jadi angka di sini **langsung sebanding dengan Tabel 16** naskah v2.

## Tahapan

| Tahap | Isi | Jumlah run | Perkiraan waktu |
|---|---|---|---|
| **Tahap 1** (`RUN_STAGE1=True`) | CTGAN + CNN-LSTM **tanpa class weight**, N=5000, 5 seed. **Sudah selesai di v2** → otomatis dilewati (hanya tabel/analisis yang dihitung ulang, ± 2–4 menit). | 5 pelatihan (sudah ada) | ± 0 (dilewati) |
| **Tahap 1b** (`RUN_STAGE1B=True`) — **BARU** | **SMOTE + CNN-LSTM tanpa class weight**, 5 seed. Data latih identik dengan baseline "SMOTE + class weights" di `results/` (Web Attack 290 → 290+5000 dengan SMOTE k=5, lalu undersample Benign); satu-satunya beda adalah class weight dimatikan. Tujuannya menghapus faktor perancu: CTGAN tanpa class weight sebelumnya dibandingkan dengan SMOTE **dengan** class weight. Lalu mean ± std, uji t berpasangan + Wilcoxon dengan koreksi Holm, dan filter LightGBM tahap kedua. | 5 pelatihan | L4: ± 1,5–2 jam · A100: ± 1–1,75 jam |
| **Tahap 2** (`RUN_STAGE2=False`) | Opsional, CTGAN dengan kolom diskret. Tidak perlu dijalankan sekarang. | 5 fit CTGAN + 10 pelatihan | ± 2–4 jam |

## Folder di Google Drive

```
MyDrive/Colab Notebooks/MECS_Revisi/
  02-14-2018.csv, 02-16-2018.csv, 02-22-2018.csv   (boleh juga di subfolder Data/)
  results/      ← hasil lama, HANYA DIBACA (tidak pernah ditulis oleh notebook ini)
  results_v2/   ← semua keluaran baru (Tahap 1 dari v2 sudah ada di sini; Tahap 1b ditambahkan ke sini)
```

## Cara menjalankan

1. Upload notebook ini ke folder `MECS_Revisi`, buka di Colab.
2. **Runtime → Change runtime type → GPU** (A100 bila tersedia; L4 juga boleh — GPU yang dipakai dicatat otomatis di `results_v2/environment.json`).
3. **Runtime → Run all**. Tidak ada yang perlu diubah. Tahap 1 dilewati otomatis, Tahap 1b berjalan.
4. Jika sesi terputus: buka lagi dan **Run all** lagi. Run yang sudah `status=ok` (di `results_v2/all_runs_v2.csv` atau `results_v2/run_logs/`) dilewati otomatis.
5. Kirim isi folder `results_v2/tables/`, `results_v2/figures/`, `results_v2/all_runs_v2.csv`, dan `results_v2/environment.json`.

In [1]:
# ============================================================
# KONFIGURASI + mount Google Drive
# Biasanya TIDAK ADA yang perlu diubah di sel ini.
# Tahap 1 sudah selesai (v2) dan akan dilewati otomatis; Tahap 1b berjalan.
# ============================================================
from pathlib import Path
import os
import sys
import shutil
import json

NOTEBOOK_VERSION = "v3"
RUN_STAGE1 = True    # Tahap 1: CTGAN asli + CNN-LSTM TANPA class weight, 5 seed (sudah selesai -> dilewati otomatis)
RUN_STAGE1B = True   # Tahap 1b (BARU): SMOTE + CNN-LSTM TANPA class weight, 5 seed
RUN_STAGE2 = False   # Tahap 2: CTGAN dengan kolom diskret. Nyalakan HANYA jika Tahap 1 lemah.

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/MECS_Revisi")
EXPECTED_CSVS = ["02-14-2018.csv", "02-16-2018.csv", "02-22-2018.csv"]
COPY_CSV_TO_LOCAL = True
LOCAL_DATA_DIR = Path("/content/cic_data")

# Protokol (sama dengan mode full notebook sebelumnya — jangan diubah agar sebanding dengan Tabel 16)
SEEDS = [42, 123, 456, 789, 1024]
N_SYNTH = 5000
SPLIT_SEED = 42
TEST_SIZE = 0.20
DL_EPOCHS = 50
DL_BATCH_SIZE = 512
DL_PATIENCE = 5
USE_MIXED_PRECISION = False   # float32, sama dengan run sebelumnya
CTGAN_EPOCHS = 200
CTGAN_BATCH_SIZE = 50
CTGAN_PAC = 10
CTGAN_GENERATOR_DIM = (256, 256)
CTGAN_DISCRIMINATOR_DIM = (256, 256)
THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

# Tahap 2: kriteria kolom diskret (sama dengan analisis sebelumnya: <=10 nilai unik
# pada 290 baris Web Attack train -> 25 kolom: kolom konstan, port, protokol, flag)
DISCRETE_MAX_UNIQUE = 10
DISCRETE_FORCE_KEYWORDS = ["port", "protocol", "flag"]
STAGE2_CLIP_TO_REAL_RANGE = False   # True = potong sampel ke rentang min/max data asli (opsional)
RUN_TSTR_STAGE2 = True              # TSTR memakai Random Forest pada ~2,5 juta baris (± 10–25 menit CPU)

# ------------------------------------------------------------
# Mode uji (smoke) — HANYA untuk pengujian di luar Colab.
# Di Colab variabel SMOKE_OVERRIDE tidak ada, jadi bagian ini tidak aktif.
# ------------------------------------------------------------
SMOKE_OVERRIDE = globals().get("SMOKE_OVERRIDE")
SMOKE = bool(SMOKE_OVERRIDE)
if SMOKE:
    print("*** MODE SMOKE: data palsu, folder sementara. JANGAN dipakai untuk naskah. ***")
    DRIVE_ROOT = Path(SMOKE_OVERRIDE["drive_root"])
    LOCAL_DATA_DIR = Path(SMOKE_OVERRIDE["local_data_dir"])
    SEEDS = list(SMOKE_OVERRIDE.get("seeds", [42, 123, 456]))
    N_SYNTH = int(SMOKE_OVERRIDE.get("n_synth", 60))
    DL_EPOCHS = int(SMOKE_OVERRIDE.get("dl_epochs", 2))
    DL_BATCH_SIZE = int(SMOKE_OVERRIDE.get("dl_batch", 64))
    DL_PATIENCE = int(SMOKE_OVERRIDE.get("dl_patience", 1))
    CTGAN_EPOCHS = int(SMOKE_OVERRIDE.get("ctgan_epochs", 2))
    RUN_STAGE1 = bool(SMOKE_OVERRIDE.get("run_stage1", True))
    RUN_STAGE1B = bool(SMOKE_OVERRIDE.get("run_stage1b", True))
    RUN_STAGE2 = bool(SMOKE_OVERRIDE.get("run_stage2", True))

OLD_RESULTS_DIR = DRIVE_ROOT / "results"        # HANYA DIBACA
RESULTS_V2_DIR = DRIVE_ROOT / "results_v2"      # semua keluaran baru
RUN_DIR = RESULTS_V2_DIR / "runs"               # history, CM, proba per run
RUN_LOG_DIR = RESULTS_V2_DIR / "run_logs"
TABLE_DIR = RESULTS_V2_DIR / "tables"
FIG_DIR = RESULTS_V2_DIR / "figures"
SYNTH_DIR = RESULTS_V2_DIR / "synth_cache"
ALL_RUNS_V2_CSV = RESULTS_V2_DIR / "all_runs_v2.csv"
# alias agar fungsi salinan dari notebook lama menulis ke results_v2, bukan results
RESULTS_DIR = RESULTS_V2_DIR

assert OLD_RESULTS_DIR.resolve() != RESULTS_V2_DIR.resolve()
assert OLD_RESULTS_DIR.resolve() not in RESULTS_V2_DIR.resolve().parents


def guard_write(path):
    """Tolak penulisan apa pun ke folder results/ lama."""
    p = Path(path).resolve()
    old = OLD_RESULTS_DIR.resolve()
    if p == old or old in p.parents:
        raise PermissionError(f"Dilarang menulis ke folder hasil lama: {p}")
    return Path(path)


# Mount Drive (aman dijalankan ulang)
try:
    from google.colab import drive as _colab_drive
    IN_COLAB = True
except ImportError:
    _colab_drive = None
    IN_COLAB = False
if IN_COLAB and not SMOKE:
    if (Path("/content/drive") / "MyDrive").exists():
        print("Google Drive sudah ter-mount.")
    else:
        _colab_drive.mount("/content/drive")
elif not SMOKE:
    print("Bukan runtime Google Colab. Pastikan DRIVE_ROOT menunjuk ke folder yang benar.")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f"Folder DRIVE_ROOT belum ada: {DRIVE_ROOT}")
if not OLD_RESULTS_DIR.exists():
    print(f"PERINGATAN: folder hasil lama tidak ditemukan: {OLD_RESULTS_DIR}. "
          "Perbandingan dengan model utama/SMOTE akan dilewati, dan CTGAN akan dilatih ulang.")

for _p in (RESULTS_V2_DIR, RUN_DIR, RUN_LOG_DIR, TABLE_DIR, FIG_DIR, SYNTH_DIR):
    guard_write(_p).mkdir(parents=True, exist_ok=True)


def snapshot_old_results():
    """Catat nama, ukuran, dan waktu ubah setiap berkas di results/ (untuk membuktikan tidak ada yang ditulis)."""
    snap = {}
    if OLD_RESULTS_DIR.exists():
        for p in OLD_RESULTS_DIR.rglob("*"):
            if p.is_file():
                st = p.stat()
                snap[str(p.relative_to(OLD_RESULTS_DIR))] = [st.st_size, st.st_mtime]
    return snap


if "OLD_RESULTS_SNAPSHOT" not in globals():
    OLD_RESULTS_SNAPSHOT = snapshot_old_results()
print(f"Folder hasil lama berisi {len(OLD_RESULTS_SNAPSHOT)} berkas (hanya dibaca).")

# ---- Cari tiga CSV: langsung di DRIVE_ROOT, atau di DRIVE_ROOT/Data ----
def find_csv_folder():
    for folder in (DRIVE_ROOT, DRIVE_ROOT / "Data"):
        if not folder.exists():
            continue
        by_lower = {p.name.lower(): p for p in folder.iterdir() if p.is_file()}
        mapping = {n: by_lower.get(n.lower()) for n in EXPECTED_CSVS}
        if all(mapping.values()):
            return folder, mapping
    return None, {}


src_folder, csv_map = find_csv_folder()
local_ok = all((LOCAL_DATA_DIR / n).exists() for n in EXPECTED_CSVS)
if src_folder is None and not local_ok:
    raise FileNotFoundError(
        "Tiga CSV (" + ", ".join(EXPECTED_CSVS) + ") tidak ditemukan di "
        f"{DRIVE_ROOT} maupun {DRIVE_ROOT / 'Data'}."
    )
if COPY_CSV_TO_LOCAL and src_folder is not None:
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name, src in csv_map.items():
        dst = LOCAL_DATA_DIR / name
        if (not dst.exists()) or dst.stat().st_size != src.stat().st_size:
            print(f"Menyalin {src} -> {dst} ...")
            shutil.copy2(src, dst)
    DATA_DIR = LOCAL_DATA_DIR
elif src_folder is not None:
    DATA_DIR = src_folder
else:
    DATA_DIR = LOCAL_DATA_DIR
    print("Memakai salinan lokal yang sudah ada.")
for name in EXPECTED_CSVS:
    f = Path(DATA_DIR) / name
    print(f"  {name}: {f} ({f.stat().st_size:,} byte)")

print("DRIVE_ROOT      =", DRIVE_ROOT)
print("DATA_DIR        =", DATA_DIR)
print("OLD_RESULTS_DIR =", OLD_RESULTS_DIR, "(hanya dibaca)")
print("RESULTS_V2_DIR  =", RESULTS_V2_DIR)
print(f"Notebook {NOTEBOOK_VERSION} | RUN_STAGE1={RUN_STAGE1} | RUN_STAGE1B={RUN_STAGE1B} | RUN_STAGE2={RUN_STAGE2} | seeds={SEEDS} | N_SYNTH={N_SYNTH} | "
      f"epoch={DL_EPOCHS} | patience={DL_PATIENCE} | batch={DL_BATCH_SIZE}")

Mounted at /content/drive
Folder hasil lama berisi 241 berkas (hanya dibaca).
Menyalin /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/Data/02-14-2018.csv -> /content/cic_data/02-14-2018.csv ...
Menyalin /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/Data/02-16-2018.csv -> /content/cic_data/02-16-2018.csv ...
Menyalin /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/Data/02-22-2018.csv -> /content/cic_data/02-22-2018.csv ...
  02-14-2018.csv: /content/cic_data/02-14-2018.csv (358,223,333 byte)
  02-16-2018.csv: /content/cic_data/02-16-2018.csv (333,723,605 byte)
  02-22-2018.csv: /content/cic_data/02-22-2018.csv (382,636,202 byte)
DRIVE_ROOT      = /content/drive/MyDrive/Colab Notebooks/MECS_Revisi
DATA_DIR        = /content/cic_data
OLD_RESULTS_DIR = /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/results (hanya dibaca)
RESULTS_V2_DIR  = /content/drive/MyDrive/Colab Notebooks/MECS_Revisi/results_v2
Notebook v3 | RUN_STAGE1=True | RUN_STAGE1B=True | RUN_STAGE2=False | seeds

In [2]:
# ============================================================
# Instal pustaka tambahan (TensorFlow sudah ada di Colab — jangan di-pin ulang)
# ============================================================
import subprocess
if IN_COLAB and not SMOKE:
    pkgs = ["ctgan", "imbalanced-learn", "lightgbm", "psutil"]
    print("Memasang:", ", ".join(pkgs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    print("Instalasi selesai. Jika import di sel berikutnya gagal, pilih Runtime → Restart session, "
          "lalu jalankan lagi semua sel dari atas.")
else:
    print("Lewati pip install (bukan Colab / mode smoke).")

Memasang: ctgan, imbalanced-learn, lightgbm, psutil
Instalasi selesai. Jika import di sel berikutnya gagal, pilih Runtime → Restart session, lalu jalankan lagi semua sel dari atas.


In [3]:
# ============================================================
# Import
# ============================================================
import gc
import time
import random
import platform
import traceback
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
if SMOKE:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set(style="whitegrid")

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score,
    precision_recall_curve, average_precision_score, roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from scipy import stats as scipy_stats

try:
    import torch
except ImportError:
    torch = None

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

try:
    from ctgan import CTGAN
    CTGAN_AVAILABLE = True
except ImportError as exc:
    CTGAN = None
    CTGAN_AVAILABLE = False
    print("CTGAN belum terpasang:", exc)

try:
    from imblearn.over_sampling import SMOTE
    IMBLEARN_AVAILABLE = True
except ImportError as exc:
    SMOTE = None
    IMBLEARN_AVAILABLE = False
    print("imbalanced-learn belum terpasang:", exc)

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if isinstance(obj, pd.DataFrame) else obj)

print("Import selesai. TF", tf.__version__, "| CTGAN:", CTGAN_AVAILABLE, "| imblearn:", IMBLEARN_AVAILABLE,
      "| LightGBM:", LGBM_AVAILABLE)

Import selesai. TF 2.21.0 | CTGAN: True | imblearn: True | LightGBM: True


In [4]:
# ============================================================
# Log hardware / lingkungan -> results_v2/environment.json
# ============================================================
def _run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True, timeout=30)
    except Exception as exc:
        return f"(gagal menjalankan {cmd!r}: {exc})"


def configure_tf_gpu():
    info = {"memory_growth": True, "mixed_precision": bool(USE_MIXED_PRECISION)}
    try:
        gpus = tf.config.list_physical_devices("GPU")
        info["physical_gpus"] = [g.name for g in gpus]
        for g in gpus:
            try:
                tf.config.experimental.set_memory_growth(g, True)
            except Exception as exc:
                print("set_memory_growth gagal (mungkin sudah diinisialisasi):", exc)
        tf.keras.mixed_precision.set_global_policy("float32")
        print("Mixed precision NONAKTIF (float32), agar sebanding dengan eksperimen sebelumnya.")
    except Exception as exc:
        info["tf_gpu_error"] = str(exc)
    return info


def log_environment(save_path=RESULTS_V2_DIR / "environment.json"):
    info = {
        "timestamp_iso": datetime.now().isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count_logical": os.cpu_count(),
        "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
        "tensorflow": tf.__version__,
        "torch": getattr(torch, "__version__", None) if torch is not None else None,
        "ctgan_available": CTGAN_AVAILABLE, "lightgbm_available": LGBM_AVAILABLE,
        "notebook_version": NOTEBOOK_VERSION,
        "smoke": SMOKE, "run_stage1": RUN_STAGE1, "run_stage1b": RUN_STAGE1B, "run_stage2": RUN_STAGE2,
        "imblearn_available": IMBLEARN_AVAILABLE,
        "seeds": SEEDS, "n_synth": N_SYNTH, "dl_epochs": DL_EPOCHS, "dl_batch_size": DL_BATCH_SIZE,
        "dl_patience": DL_PATIENCE, "ctgan_epochs": CTGAN_EPOCHS, "ctgan_batch_size": CTGAN_BATCH_SIZE,
        "ctgan_pac": CTGAN_PAC, "use_mixed_precision": USE_MIXED_PRECISION,
        "drive_root": str(DRIVE_ROOT), "data_dir": str(DATA_DIR),
        "old_results_dir_read_only": str(OLD_RESULTS_DIR), "results_v2_dir": str(RESULTS_V2_DIR),
        "in_colab": IN_COLAB,
    }
    try:
        import psutil
        vm = psutil.virtual_memory()
        info["ram_total_gb"] = round(vm.total / (1024 ** 3), 2)
    except ImportError:
        pass
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.exists():
        for line in cpuinfo.read_text(errors="replace").splitlines():
            if "model name" in line:
                info["cpu_model"] = line.split(":", 1)[1].strip()
                break
    smi = _run_cmd(["nvidia-smi"])
    print("----- nvidia-smi -----")
    print(smi)
    info["nvidia_smi"] = smi[-4000:]
    info["tf_gpus"] = [g.name for g in tf.config.list_physical_devices("GPU")]
    for i, g in enumerate(tf.config.list_physical_devices("GPU")):
        try:
            det = tf.config.experimental.get_device_details(g)
            info.setdefault("tf_gpu_details", []).append({str(k): str(v) for k, v in det.items()})
        except Exception:
            pass
    info["torch_cuda"] = bool(torch is not None and torch.cuda.is_available())
    if info["torch_cuda"]:
        info["torch_gpu_name"] = torch.cuda.get_device_name(0)
    info.update(configure_tf_gpu())
    info["gpu_name"] = detect_gpu_name(info)
    info["gpu_is_a100"] = "A100" in str(info["gpu_name"]).upper()
    # Riwayat: environment.json dari sesi sebelumnya (mis. Tahap 1 dengan v2 di L4) tidak hilang
    history = []
    if Path(save_path).exists():
        try:
            prev = json.loads(Path(save_path).read_text())
            history = list(prev.pop("previous_sessions", []))
            prev_gpu = prev.get("gpu_name") or detect_gpu_name(prev)
            history.append({"timestamp_iso": prev.get("timestamp_iso"),
                            "notebook_version": prev.get("notebook_version", "v2"),
                            "gpu_name": prev_gpu, "ram_total_gb": prev.get("ram_total_gb"),
                            "cpu_model": prev.get("cpu_model"), "cpu_count_logical": prev.get("cpu_count_logical"),
                            "run_stage1": prev.get("run_stage1"), "run_stage1b": prev.get("run_stage1b"),
                            "run_stage2": prev.get("run_stage2")})
        except Exception as exc:
            print("environment.json lama tidak terbaca:", exc)
    info["previous_sessions"] = history
    guard_write(save_path).write_text(json.dumps(info, indent=2))
    print(json.dumps({k: v for k, v in info.items() if k != "nvidia_smi"}, indent=2))
    print("Tersimpan:", save_path)
    print(f"GPU yang dipakai sesi ini: {info['gpu_name']}")
    if history:
        print("GPU sesi-sesi sebelumnya di results_v2:", [h.get("gpu_name") for h in history])
    if not info["tf_gpus"] and not SMOKE:
        print("PERINGATAN: TensorFlow tidak melihat GPU. Pilih Runtime → Change runtime type → GPU.")
    if not info["gpu_is_a100"]:
        print("!" * 72)
        print(f"PERHATIAN: GPU sesi ini adalah '{info['gpu_name']}', BUKAN A100 seperti run di results/.")
        print("Akurasi/F1 tidak terpengaruh, tetapi bagian perangkat keras di naskah harus menyebut GPU ini,")
        print("dan waktu pelatihan/inferensi dari GPU berbeda tidak boleh dicampur dalam satu tabel.")
        print("Nama GPU juga dicatat per run di kolom 'gpu_name' pada all_runs_v2.csv.")
        print("!" * 72)
    return info


def detect_gpu_name(info):
    """Nama GPU dari torch, detail TF, atau teks nvidia-smi (urutan prioritas)."""
    if info.get("torch_gpu_name"):
        return str(info["torch_gpu_name"])
    for d in info.get("tf_gpu_details", []) or []:
        if d.get("device_name"):
            return str(d["device_name"])
    smi = str(info.get("nvidia_smi", ""))
    import re as _re
    m = _re.search(r"NVIDIA\s+([A-Za-z0-9][A-Za-z0-9 \-]*?)\s{2,}", smi)
    if m:
        return "NVIDIA " + m.group(1).strip()
    return "tidak ada GPU (CPU)"


ENV_INFO = log_environment()
GPU_NAME = ENV_INFO.get("gpu_name")

----- nvidia-smi -----
Sat Nov 14 21:33:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+------------------------

## Praproses — disalin apa adanya dari notebook sebelumnya

Fungsi pemuat, pembersihan, split 80/20 (`random_state=42`, `stratify=y`), MinMax (fit hanya pada train), dan undersample Benign identik dengan notebook `revisi_eksperimen_colab.ipynb`. Seed lain hanya memengaruhi CTGAN, undersample, dan inisialisasi model, **bukan** pecahan test.

In [5]:
# ============================================================
# Fungsi praproses (identik dengan notebook sebelumnya)
# ============================================================
KOLOM_METADATA = ["flow id", "src ip", "dst ip", "src port", "timestamp"]
WEB_LABELS = ["Brute Force -Web", "Brute Force -XSS", "SQL Injection"]
CLASS_ORDER = [
    "Benign", "DoS attacks-Hulk", "DoS attacks-SlowHTTPTest",
    "FTP-BruteForce", "SSH-Bruteforce", "Web Attack",
]


def set_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    tf.random.set_seed(seed)


def load_cic_dataset(base_path: Path) -> pd.DataFrame:
    """Tahap 2 asli: muat CSV, buang metadata, downcast, buang inf/NaN/header anomali."""
    base_path = Path(base_path)
    all_files = sorted(str(base_path / n) for n in EXPECTED_CSVS)
    print(f"Memuat {len(all_files)} file CSV dari {base_path}")
    df_list = []
    for file_path in all_files:
        print(f"Memproses {os.path.basename(file_path)}...")
        temp_df = pd.read_csv(file_path, low_memory=False)
        temp_df.columns = temp_df.columns.str.strip().str.lower()
        temp_df = temp_df.drop(columns=[c for c in KOLOM_METADATA if c in temp_df.columns], errors="ignore")
        for col in temp_df.select_dtypes(include=["float64"]).columns:
            temp_df[col] = temp_df[col].astype("float32")
        for col in temp_df.select_dtypes(include=["int64"]).columns:
            temp_df[col] = pd.to_numeric(temp_df[col], downcast="integer")
        df_list.append(temp_df)
        gc.collect()
    print("Menggabungkan dataset...")
    df = pd.concat(df_list, ignore_index=True)
    del df_list
    gc.collect()
    if "label" in df.columns:
        df = df[df["label"] != "Label"]
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    if "label" in df.columns:
        df.rename(columns={"label": "Label"}, inplace=True)
    print(f"Total baris bersih: {df.shape[0]:,} | kolom: {df.shape[1]}")
    return df


def clean_and_map_multiclass(df: pd.DataFrame) -> pd.DataFrame:
    """Tahap 4–5 asli: numerik, buang kolom konstan, gabung Web Attack."""
    df = df.copy()
    df["Label"] = df["Label"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    fitur = [c for c in df.columns if c != "Label"]
    df[fitur] = df[fitur].apply(pd.to_numeric, errors="coerce")
    n_nan = int(df[fitur].isna().sum().sum())
    print(f"NaN setelah konversi numerik: {n_nan:,}")
    if n_nan > 0:
        df.dropna(subset=fitur, inplace=True)
    nunique = df[fitur].nunique(dropna=False)
    kolom_konstan = nunique[nunique <= 1].index.tolist()
    print("Kolom konstan dibuang:", kolom_konstan)
    if kolom_konstan:
        df.drop(columns=kolom_konstan, inplace=True)
    df["Label"] = df["Label"].replace({k: "Web Attack" for k in WEB_LABELS})
    print("Distribusi setelah gabung Web Attack:")
    print(df["Label"].value_counts())
    return df


def split_and_scale(X, y, split_seed=SPLIT_SEED, test_size=TEST_SIZE):
    print(f"Train-test split {1-test_size:.0%}/{test_size:.0%}, stratify=y, random_state={split_seed}")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=split_seed, stratify=y,
    )
    print(f"Train: {X_train.shape[0]:,} x {X_train.shape[1]} | Test: {X_test.shape[0]:,} x {X_test.shape[1]}")
    print("Distribusi TRAIN:\n", y_train.value_counts())
    print("Distribusi TEST (jangan diubah):\n", y_test.value_counts())
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    feature_cols = list(X_train.columns)
    X_train_df = pd.DataFrame(X_train_scaled, columns=feature_cols).reset_index(drop=True)
    X_test_df = pd.DataFrame(X_test_scaled, columns=feature_cols).reset_index(drop=True)
    return scaler, X_train_df, X_test_df, pd.Series(y_train).reset_index(drop=True), pd.Series(y_test).reset_index(drop=True)


def undersample_benign(train_df: pd.DataFrame, random_state=42) -> pd.DataFrame:
    """Seperti Tahap 12: Benign di-sample sebanyak jumlah baris non-Benign."""
    df_benign = train_df[train_df["Label"] == "Benign"]
    df_attack = train_df[train_df["Label"] != "Benign"]
    n = len(df_attack)
    if len(df_benign) < n:
        print(f"Peringatan: Benign ({len(df_benign)}) < serangan ({n}); pakai semua Benign.")
        df_b = df_benign
    else:
        df_b = df_benign.sample(n=n, random_state=random_state)
    out = pd.concat([df_b, df_attack], ignore_index=True)
    return out.sample(frac=1, random_state=random_state).reset_index(drop=True)


def attach_synth_web(X_train_df, y_train, X_web_synth) -> pd.DataFrame:
    train_combined = X_train_df.copy()
    train_combined["Label"] = y_train.values
    synth_df = X_web_synth.copy()
    synth_df["Label"] = "Web Attack"
    return pd.concat([train_combined, synth_df], ignore_index=True)


def build_train_main_ctgan(X_synth, seed):
    """Sama dengan skenario main_ctgan: train + N sintetis Web Attack, lalu undersample Benign (seed)."""
    combined = attach_synth_web(X_train_mm, y_train, X_synth)
    bal = undersample_benign(combined, random_state=seed)
    print("Distribusi train (main_ctgan):\n", bal["Label"].value_counts())
    return bal.drop(columns=["Label"]), bal["Label"]


def oversample_web_smote(X_train_df, y_train_s, n_synth, seed):
    """Identik dengan oversample_web_imblearn(method="smote") di notebook sebelumnya:
    hanya Web Attack yang di-oversample sampai n_real + n_synth (290 + 5000), k_neighbors=min(5, n_real-1),
    random_state=seed; kelas lain tidak diubah."""
    if not IMBLEARN_AVAILABLE:
        raise ImportError("imbalanced-learn diperlukan untuk SMOTE — jalankan sel pip install.")
    y = pd.Series(y_train_s).reset_index(drop=True)
    X = X_train_df.reset_index(drop=True)
    n_real = int((y == "Web Attack").sum())
    target = n_real + int(n_synth)
    k = min(5, max(1, n_real - 1))
    sampling = {lab: int((y == lab).sum()) for lab in y.unique()}
    sampling["Web Attack"] = target
    sampler = SMOTE(k_neighbors=k, random_state=seed, sampling_strategy=sampling)
    X_res, y_res = sampler.fit_resample(X, y)
    X_res = pd.DataFrame(X_res, columns=X.columns)
    y_res = pd.Series(y_res, name="Label")
    print(f"smote: Web Attack {n_real} -> {(y_res == 'Web Attack').sum()} (target {target}, k={k}, seed={seed})")
    return X_res, y_res


def build_train_smote(seed, n_synth=None):
    """Sama dengan skenario 'smote' di results/: SMOTE Web Attack -> undersample Benign (seed)."""
    n_synth = N_SYNTH if n_synth is None else n_synth
    X_res, y_res = oversample_web_smote(X_train_mm, y_train, n_synth, seed)
    combined = X_res.copy()
    combined["Label"] = y_res.values
    bal = undersample_benign(combined, random_state=seed)
    print("Distribusi train (smote):\n", bal["Label"].value_counts())
    return bal.drop(columns=["Label"]), bal["Label"]

In [6]:
# ============================================================
# Muat data, split seed 42, MinMax (protokol sama dengan run sebelumnya)
# ============================================================
set_seeds(SPLIT_SEED)
df_raw = load_cic_dataset(DATA_DIR)
df_model = clean_and_map_multiclass(df_raw)
del df_raw
gc.collect()
feature_cols = [c for c in df_model.columns if c != "Label"]
print(f"Jumlah fitur: {len(feature_cols)} (run sebelumnya: 68)")
X = df_model[feature_cols]
y = df_model["Label"]
scaler, X_train_mm, X_test_mm, y_train, y_test = split_and_scale(X, y, split_seed=SPLIT_SEED)
del df_model, X
gc.collect()

X_web_gan = X_train_mm[y_train.values == "Web Attack"].copy().reset_index(drop=True)
print(f"Bahan CTGAN Web Attack: {X_web_gan.shape[0]} baris x {X_web_gan.shape[1]} fitur (run sebelumnya: 290 x 68)")
X_web_gan.to_csv(guard_write(RESULTS_V2_DIR / "X_web_gan.csv"), index=False)

# Cek kecocokan dengan run sebelumnya bila file lama tersedia (hanya dibaca)
_old_gan = OLD_RESULTS_DIR / "X_web_gan.csv"
if _old_gan.exists():
    _o = pd.read_csv(_old_gan)
    same = (_o.shape == X_web_gan.shape) and list(_o.columns) == list(X_web_gan.columns) and \
        np.allclose(_o.to_numpy(dtype=float), X_web_gan.to_numpy(dtype=float), atol=1e-6)
    print("X_web_gan identik dengan run sebelumnya:", same)
    if not same:
        print("PERINGATAN: praproses tidak identik dengan run sebelumnya. Hasil mungkin tidak sebanding!")

Memuat 3 file CSV dari /content/cic_data
Memproses 02-14-2018.csv...
Memproses 02-16-2018.csv...
Memproses 02-22-2018.csv...
Menggabungkan dataset...
Total baris bersih: 3,136,290 | kolom: 79
NaN setelah konversi numerik: 0
Kolom konstan dibuang: ['bwd psh flags', 'fwd urg flags', 'bwd urg flags', 'cwe flag count', 'fwd byts/b avg', 'fwd pkts/b avg', 'fwd blk rate avg', 'bwd byts/b avg', 'bwd pkts/b avg', 'bwd blk rate avg']
Distribusi setelah gabung Web Attack:
Label
Benign                      2153183
DoS attacks-Hulk             461912
FTP-BruteForce               193354
SSH-Bruteforce               187589
DoS attacks-SlowHTTPTest     139890
Web Attack                      362
Name: count, dtype: int64
Jumlah fitur: 68 (run sebelumnya: 68)
Train-test split 80%/20%, stratify=y, random_state=42
Train: 2,509,032 x 68 | Test: 627,258 x 68
Distribusi TRAIN:
 Label
Benign                      1722546
DoS attacks-Hulk             369530
FTP-BruteForce               154683
SSH-Bruteforce   

In [7]:
# ============================================================
# CTGAN (opsional kolom diskret), mutu sintetis, TSTR
# ============================================================
def fit_ctgan_and_sample(X_real, n_synth, seed, tag, discrete_columns=None, epochs=None):
    """Latih CTGAN hanya pada Web Attack train, lalu sample n_synth baris (setelan sama dengan run sebelumnya)."""
    if not CTGAN_AVAILABLE:
        raise RuntimeError("CTGAN tidak tersedia — jalankan sel pip install.")
    set_seeds(seed)
    epochs = CTGAN_EPOCHS if epochs is None else epochs
    pac = CTGAN_PAC
    batch_size = CTGAN_BATCH_SIZE
    if batch_size >= len(X_real):
        batch_size = max(pac, (len(X_real) // pac) * pac)
        print(f"batch_size CTGAN disesuaikan ke {batch_size}")
    discrete_columns = list(discrete_columns or [])
    t0 = time.perf_counter()
    model = CTGAN(epochs=epochs, batch_size=batch_size, generator_dim=CTGAN_GENERATOR_DIM,
                  discriminator_dim=CTGAN_DISCRIMINATOR_DIM, pac=pac, verbose=False)
    model.fit(X_real, discrete_columns=discrete_columns)
    fit_s = time.perf_counter() - t0
    set_seeds(seed)
    X_synth = model.sample(n_synth)
    X_synth.columns = X_real.columns
    X_synth = X_synth.astype(np.float64)
    print(f"CTGAN[{tag}] seed={seed}: fit {fit_s:.1f}s, {len(discrete_columns)} kolom diskret, sampel {X_synth.shape}")
    try:
        loss_df = model.loss_values.copy()
        loss_df.to_csv(guard_write(SYNTH_DIR / f"ctgan_loss_{tag}_seed{seed}.csv"), index=False)
    except Exception as exc:
        print("Loss CTGAN tidak tersimpan:", exc)
    return X_synth, {"fit_seconds": fit_s, "epochs": epochs, "batch_size": batch_size,
                     "n_discrete": len(discrete_columns), "tag": tag}


def _validate_synth(Xs, n_synth, source):
    missing = [c for c in X_web_gan.columns if c not in Xs.columns]
    if missing:
        raise ValueError(f"Kolom sintetis tidak cocok ({source}); hilang: {missing[:5]}")
    if len(Xs) < n_synth:
        raise ValueError(f"Sintetis di {source} hanya {len(Xs)} baris, perlu {n_synth}")
    return Xs[list(X_web_gan.columns)].iloc[:n_synth].astype(np.float64).reset_index(drop=True)


def get_synth_original(seed, n_synth=None):
    """CTGAN asli (tanpa kolom diskret). Pakai cache run sebelumnya dari results/ (dibaca saja)."""
    n_synth = N_SYNTH if n_synth is None else n_synth
    name = f"X_web_synth_n{n_synth}_seed{seed}_n{n_synth}.csv"   # penamaan notebook sebelumnya
    for cand in (OLD_RESULTS_DIR / name, SYNTH_DIR / name):
        if cand.exists():
            print(f"Memakai sintetis CTGAN ter-cache: {cand}")
            return _validate_synth(pd.read_csv(cand), n_synth, cand), {"cached": True, "synth_file": str(cand)}
    print(f"Cache {name} tidak ada di results/ — melatih CTGAN (setelan sama), disimpan ke results_v2.")
    Xs, meta = fit_ctgan_and_sample(X_web_gan, n_synth, seed, tag=f"n{n_synth}")
    out = guard_write(SYNTH_DIR / name)
    Xs.to_csv(out, index=False)
    return Xs, {**meta, "cached": False, "synth_file": str(out)}


def select_discrete_columns(X_real):
    """Kolom dengan <= DISCRETE_MAX_UNIQUE nilai unik (termasuk konstan), plus port/protokol/flag."""
    nun = X_real.nunique()
    cols = []
    for c in X_real.columns:
        if nun[c] <= DISCRETE_MAX_UNIQUE or any(k in c.lower() for k in DISCRETE_FORCE_KEYWORDS):
            cols.append(c)
    return cols


def get_synth_discrete(seed, n_synth=None):
    n_synth = N_SYNTH if n_synth is None else n_synth
    clip_tag = "_clip" if STAGE2_CLIP_TO_REAL_RANGE else ""
    cache = SYNTH_DIR / f"X_web_synth_discrete{clip_tag}_n{n_synth}_seed{seed}.csv"
    if cache.exists():
        print(f"Memakai sintetis CTGAN-diskret ter-cache: {cache}")
        return _validate_synth(pd.read_csv(cache), n_synth, cache), {"cached": True, "synth_file": str(cache)}
    Xs, meta = fit_ctgan_and_sample(X_web_gan, n_synth, seed, tag=f"discrete{clip_tag}",
                                    discrete_columns=DISCRETE_COLS)
    if STAGE2_CLIP_TO_REAL_RANGE:
        Xs = Xs.clip(lower=X_web_gan.min(), upper=X_web_gan.max(), axis=1)
    Xs.to_csv(guard_write(cache), index=False)
    return Xs, {**meta, "cached": False, "synth_file": str(cache)}


def evaluate_synth_quality(X_real, X_synth, tag, seed=42):
    """KS per fitur, diskriminator real-vs-sintetis (RF, AUC), selisih korelasi — sama dengan run sebelumnya."""
    cols = list(X_real.columns)
    real = X_real[cols].astype(float)
    synth = X_synth[cols].astype(float)
    diff = (real.corr().fillna(0) - synth.corr().fillna(0)).abs()
    ks_rows = []
    for c in cols:
        try:
            stat, p = scipy_stats.ks_2samp(real[c].to_numpy(), synth[c].to_numpy())
        except Exception:
            stat, p = np.nan, np.nan
        ks_rows.append({"feature": c, "ks_statistic": stat, "p_value": p,
                        "real_mean": real[c].mean(), "synth_mean": synth[c].mean(),
                        "real_nunique": int(real[c].nunique()), "synth_nunique": int(synth[c].nunique())})
    ks_df = pd.DataFrame(ks_rows).sort_values("ks_statistic", ascending=False)
    rng = np.random.default_rng(seed)
    n = min(len(real), len(synth), 4000)
    ri = rng.choice(len(real), n, replace=False)
    si = rng.choice(len(synth), n, replace=False)
    Xd = pd.concat([real.iloc[ri], synth.iloc[si]], ignore_index=True)
    yd = np.array([1] * n + [0] * n)
    Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=seed, stratify=yd)
    clf = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1).fit(Xd_tr, yd_tr)
    pr = clf.predict_proba(Xd_te)[:, 1]
    arr = synth.to_numpy()
    rec = {
        "tag": tag, "seed": seed, "n_real": len(real), "n_synth": len(synth),
        "ks_mean": float(ks_df["ks_statistic"].mean()), "ks_median": float(ks_df["ks_statistic"].median()),
        "ks_max": float(ks_df["ks_statistic"].max()),
        "n_features_ks_gt_0_5": int((ks_df["ks_statistic"] > 0.5).sum()),
        "corr_absdiff_mean": float(diff.to_numpy()[np.triu_indices_from(diff, k=1)].mean()) if len(cols) > 1 else 0.0,
        "discriminator_auc": float(roc_auc_score(yd_te, pr)),
        "discriminator_acc": float(accuracy_score(yd_te, (pr >= 0.5).astype(int))),
        "pct_values_outside_0_1": float(((arr < 0) | (arr > 1)).mean() * 100),
    }
    print(f"Mutu [{tag}]: KS rata-rata={rec['ks_mean']:.4f}, AUC diskriminator={rec['discriminator_auc']:.4f}, "
          f"di luar [0,1]={rec['pct_values_outside_0_1']:.2f}%")
    return rec, ks_df


def run_tstr(X_synth, tag, seed=SPLIT_SEED):
    """TSTR seperti notebook sebelumnya: RF dilatih pada non-Web train asli + Web sintetis, diuji pada test asli."""
    X_neg = X_train_mm[y_train.values != "Web Attack"]
    X_tstr = pd.concat([X_neg, X_synth[X_neg.columns]], ignore_index=True)
    y_tstr = pd.concat([y_train[y_train.values != "Web Attack"],
                        pd.Series(["Web Attack"] * len(X_synth))], ignore_index=True)
    rf = RandomForestClassifier(n_estimators=150, random_state=seed, n_jobs=-1, class_weight="balanced")
    t0 = time.perf_counter()
    rf.fit(X_tstr, y_tstr)
    pred = rf.predict(X_test_mm)
    le_tmp = LabelEncoder().fit(sorted(set(y_tstr) | set(y_test)))
    rec, _ = metrics_from_preds(le_tmp.transform(y_test), le_tmp.transform(pred), list(le_tmp.classes_))
    rec["fit_seconds"] = time.perf_counter() - t0
    rec["tag"] = tag
    print(f"TSTR [{tag}] macro F1={rec['f1_macro']:.4f} | Web P/R/F1={rec.get('Web Attack__precision', 0):.4f}/"
          f"{rec.get('Web Attack__recall', 0):.4f}/{rec.get('Web Attack__f1', 0):.4f} ({rec['fit_seconds']:.0f}s)")
    return rec

In [8]:
# ============================================================
# Model CNN-LSTM (arsitektur identik) + pelatihan/evaluasi
# ============================================================
def build_model(n_timesteps: int, n_classes: int):
    model = Sequential()
    model.add(Conv1D(64, 3, activation="relu", input_shape=(n_timesteps, 1)))
    model.add(MaxPooling1D(2))
    model.add(Dropout(0.3))
    model.add(LSTM(128, return_sequences=True))
    model.add(Dropout(0.3))
    model.add(LSTM(64))
    model.add(Dense(64, activation="relu"))
    model.add(Dropout(0.3))
    model.add(Dense(n_classes, activation="softmax"))
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def metrics_from_preds(y_true_enc, y_pred_class, class_names):
    report = classification_report(y_true_enc, y_pred_class, labels=list(range(len(class_names))),
                                   target_names=class_names, digits=4, zero_division=0, output_dict=True)
    rec = {
        "accuracy": accuracy_score(y_true_enc, y_pred_class),
        "f1_macro": f1_score(y_true_enc, y_pred_class, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true_enc, y_pred_class, average="weighted", zero_division=0),
    }
    for cname in class_names:
        for k_src, k_dst in [("precision", "precision"), ("recall", "recall"), ("f1-score", "f1"), ("support", "support")]:
            rec[f"{cname}__{k_dst}"] = report[cname][k_src]
    return rec, report


def save_confusion(y_true_enc, y_pred_class, class_names, path, title):
    cm = confusion_matrix(y_true_enc, y_pred_class, labels=list(range(len(class_names))))
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(guard_write(path.with_suffix(".csv")))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xticks(rotation=35, ha="right"); plt.yticks(rotation=0)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(title)
    plt.tight_layout()
    plt.savefig(guard_write(path.with_suffix(".png")), dpi=300, bbox_inches="tight")
    plt.close()
    return cm


def train_eval_dl(X_train_df, y_train_s, use_class_weight, seed, run_name):
    set_seeds(seed)
    le = LabelEncoder()
    le.fit(CLASS_ORDER)
    present = set(pd.Series(y_train_s).unique()) | set(pd.Series(y_test).unique())
    if any(c not in le.classes_ for c in present):
        le = LabelEncoder().fit(sorted(present))
    y_train_enc = le.transform(y_train_s)
    y_test_enc = le.transform(y_test)
    n_classes = len(le.classes_)
    y_train_cat = to_categorical(y_train_enc, num_classes=n_classes)
    X_train_dl = np.asarray(X_train_df, dtype=np.float32).reshape((len(X_train_df), X_train_df.shape[1], 1))
    X_test_dl = np.asarray(X_test_mm, dtype=np.float32).reshape((len(X_test_mm), X_test_mm.shape[1], 1))

    class_weight = None
    if use_class_weight:
        weights = compute_class_weight("balanced", classes=np.unique(y_train_enc), y=y_train_enc)
        class_weight = {int(i): float(w) for i, w in zip(np.unique(y_train_enc), weights)}
        print("class_weight:", {le.classes_[i]: round(w, 3) for i, w in class_weight.items()})
    else:
        print("class_weight: TIDAK dipakai")

    model = build_model(X_train_dl.shape[1], n_classes)
    early_stop = EarlyStopping(monitor="val_loss", patience=DL_PATIENCE, restore_best_weights=True)
    t0 = time.perf_counter()
    history = model.fit(X_train_dl, y_train_cat, epochs=DL_EPOCHS, batch_size=DL_BATCH_SIZE,
                        validation_split=0.2, class_weight=class_weight, callbacks=[early_stop], verbose=2)
    train_s = time.perf_counter() - t0
    n_epochs_ran = len(history.history.get("loss", []))
    print(f"[{run_name}] pelatihan {train_s/60:.2f} menit, {n_epochs_ran} epoch (maks {DL_EPOCHS})")

    _ = model.predict(X_test_dl[:max(DL_BATCH_SIZE, 32)], batch_size=DL_BATCH_SIZE, verbose=0)  # warmup
    t1 = time.perf_counter()
    proba = model.predict(X_test_dl, batch_size=DL_BATCH_SIZE, verbose=0)
    infer_s = time.perf_counter() - t1
    y_pred = proba.argmax(axis=1)
    rec, _ = metrics_from_preds(y_test_enc, y_pred, list(le.classes_))
    rec.update({"run_name": run_name, "arch": "cnn_lstm", "seed": seed, "use_class_weight": bool(use_class_weight),
                "n_train": int(len(X_train_df)), "n_test": int(len(X_test_mm)), "train_seconds": train_s,
                "n_epochs_ran": n_epochs_ran, "inference_seconds": infer_s,
                "inference_ms_per_sample": infer_s / len(X_test_dl) * 1000.0, "smoke": SMOKE,
                "gpu_name": globals().get("GPU_NAME"), "notebook_version": NOTEBOOK_VERSION})
    print(classification_report(y_test_enc, y_pred, labels=list(range(n_classes)), target_names=le.classes_,
                                digits=4, zero_division=0))
    print(f"Macro F1={rec['f1_macro']:.4f} | Web P={rec.get('Web Attack__precision', float('nan')):.4f} "
          f"R={rec.get('Web Attack__recall', float('nan')):.4f} F1={rec.get('Web Attack__f1', float('nan')):.4f}")

    pd.DataFrame(history.history).to_csv(guard_write(RUN_DIR / f"history_{run_name}.csv"), index=False)
    save_confusion(y_test_enc, y_pred, list(le.classes_), RUN_DIR / f"cm_{run_name}",
                   f"Confusion matrix — {run_name} (test untouched)")
    proba_path = guard_write(RUN_DIR / f"proba_{run_name}.npz")
    np.savez_compressed(proba_path, proba=proba.astype(np.float32), y_true=y_test_enc, classes=np.array(le.classes_))
    rec["proba_path"] = str(proba_path)
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return rec

In [9]:
# ============================================================
# Runner dengan RESUME (results_v2/all_runs_v2.csv)
# ============================================================
def load_runs_v2():
    if ALL_RUNS_V2_CSV.exists():
        df = pd.read_csv(ALL_RUNS_V2_CSV)
        n_ok = int(df["status"].eq("ok").sum()) if "status" in df.columns else 0
        print(f"Memuat {ALL_RUNS_V2_CSV.name}: {len(df)} baris, status ok={n_ok}")
        return df.to_dict(orient="records")
    print("Belum ada all_runs_v2.csv — mulai dari awal.")
    return []


def persist_runs(rows):
    df = pd.DataFrame(rows)
    if len(df):
        df["_ok"] = df["status"].eq("ok").astype(int)
        df = df.sort_values("_ok", kind="stable").drop_duplicates("run_name", keep="last").drop(columns=["_ok"])
    df.to_csv(guard_write(ALL_RUNS_V2_CSV), index=False)
    return df


def _ok_from_run_log(run_name):
    """Cadangan resume: jika run_logs/<run>.json berstatus ok tetapi barisnya belum ada di all_runs_v2.csv
    (mis. sesi terputus tepat setelah log ditulis), pulihkan baris itu — run TIDAK dilatih ulang."""
    path = RUN_LOG_DIR / f"{run_name}.json"
    if not path.exists():
        return None
    try:
        rec = json.loads(path.read_text())
    except Exception:
        return None
    if str(rec.get("status", "")).lower() != "ok":
        return None
    rec.pop("logged_iso", None)
    print(f"Memulihkan {run_name} dari run_logs/ (tidak dilatih ulang).")
    ALL_ROWS.append(rec)
    persist_runs(ALL_ROWS)
    return rec


def find_ok(run_name):
    for r in ALL_ROWS:
        if r.get("run_name") == run_name and str(r.get("status", "")).lower() == "ok":
            return r
    return _ok_from_run_log(run_name)


def write_run_log(run_name, payload):
    clean = {}
    for k, v in {**payload, "logged_iso": datetime.now().isoformat(timespec="seconds")}.items():
        try:
            json.dumps(v)
            clean[k] = v
        except TypeError:
            clean[k] = str(v)
    guard_write(RUN_LOG_DIR / f"{run_name}.json").write_text(json.dumps(clean, indent=2))


def make_job(stage, synth_kind, use_class_weight, seed):
    """synth_kind: 'original' (CTGAN asli), 'discrete' (CTGAN kolom diskret), 'smote' (Tahap 1b)."""
    if synth_kind == "smote":
        # Tahap 1b: disimpan sebagai stage=1 + synth_kind='smote' (kolom stage tetap bilangan bulat)
        name = f"s1b_smote_cnn_lstm_w{int(use_class_weight)}_n{N_SYNTH}_seed{seed}"
        return dict(run_name=name, stage=stage, stage_label="1b", synth_kind="smote",
                    use_class_weight=use_class_weight, seed=seed)
    prefix = {"original": "s1_main_ctgan", "discrete": "s2_main_ctgan_discrete"}[synth_kind]
    if stage == 2 and synth_kind == "original":
        prefix = "s2_main_ctgan"
    name = f"{prefix}_cnn_lstm_w{int(use_class_weight)}_n{N_SYNTH}_seed{seed}"
    return dict(run_name=name, stage=stage, stage_label=str(stage), synth_kind=synth_kind,
                use_class_weight=use_class_weight, seed=seed)


def run_job(job, progress):
    name = job["run_name"]
    print("\n" + "=" * 72)
    print(f"[{progress}] RUN {name}")
    print("=" * 72)
    if find_ok(name) is not None:
        print(f"[{progress}] SKIP — sudah status=ok di all_runs_v2.csv")
        return find_ok(name)
    t_start = time.perf_counter()
    try:
        if job["synth_kind"] == "smote":
            Xb, yb = build_train_smote(job["seed"])
            smeta, scenario = {"method": "smote"}, "smote"
        else:
            if job["synth_kind"] == "original":
                Xs, smeta = get_synth_original(job["seed"])
            else:
                Xs, smeta = get_synth_discrete(job["seed"])
            Xb, yb = build_train_main_ctgan(Xs, job["seed"])
            scenario = "main_ctgan"
        rec = train_eval_dl(Xb, yb, job["use_class_weight"], job["seed"], name)
        rec.update({"stage": job["stage"], "stage_label": job.get("stage_label", str(job["stage"])),
                    "scenario": scenario, "synth_kind": job["synth_kind"],
                    "n_synth": N_SYNTH, "status": "ok", "wall_seconds": time.perf_counter() - t_start})
        rec.update({(k if k.startswith("synth_") else f"synth_{k}"): v for k, v in smeta.items()})
    except Exception as exc:
        traceback.print_exc()
        rec = {**job, "status": "error", "error": str(exc)}
    ALL_ROWS.append(rec)
    persist_runs(ALL_ROWS)
    write_run_log(name, rec)
    return rec


def run_jobs(jobs, label):
    done = sum(1 for j in jobs if find_ok(j["run_name"]) is not None)
    print(f"{label}: {len(jobs)} run terencana, sudah selesai {done}, sisa {len(jobs) - done}.")
    t0 = time.perf_counter()
    for i, job in enumerate(jobs, start=1):
        run_job(job, progress=f"{label} — run {i} of {len(jobs)}")
        print(f"Waktu berjalan {label}: {(time.perf_counter() - t0)/60:.1f} menit")


ALL_ROWS = load_runs_v2()

Memuat all_runs_v2.csv: 5 baris, status ok=5


In [10]:
# ============================================================
# Fungsi statistik: ringkasan mean ± std, uji berpasangan, koreksi Holm
# ============================================================
METRICS = ["f1_macro", "accuracy", "Web Attack__precision", "Web Attack__recall", "Web Attack__f1"]
TEST_METRICS = ["f1_macro", "Web Attack__precision", "Web Attack__recall", "Web Attack__f1"]
METRIC_LABEL = {"f1_macro": "Macro F1", "accuracy": "Accuracy", "Web Attack__precision": "WA precision",
                "Web Attack__recall": "WA recall", "Web Attack__f1": "WA F1"}


def _boolish(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes", "1.0"])


def load_old_reference():
    """Baca results/all_runs.csv (HANYA DIBACA) -> dict nama referensi -> DataFrame per seed."""
    path = OLD_RESULTS_DIR / "all_runs.csv"
    if not path.exists():
        print("results/all_runs.csv tidak ada — perbandingan dengan run lama dilewati.")
        return {}
    old = pd.read_csv(path)
    old = old[old["status"].eq("ok")] if "status" in old.columns else old
    # run tanpa akhiran tag (_ablation, _noweight, _nablate, ...) = nama berakhir dengan _seed<angka>
    tags = ~old["run_name"].astype(str).str.fullmatch(r".*_seed\d+")
    base = old[(old["arch"] == "cnn_lstm") & _boolish(old["use_class_weight"]) &
               (old["n_synth"] == N_SYNTH) & ~tags & old["seed"].isin(SEEDS)]
    refs = {}
    for scen, label in [("main_ctgan", "Targeted CTGAN + class weights (Tabel 16, results/)"),
                        ("smote", "SMOTE + class weights (results/)"),
                        ("weights_only", "No augmentation, class weights only (results/)"),
                        ("undersample_only", "Undersample only (results/)")]:
        g = base[base["scenario"] == scen].drop_duplicates("seed")
        if len(g):
            refs[scen] = g.assign(model_label=label)
    old_nw = old[old["run_name"].astype(str) == f"main_ctgan_cnn_lstm_w0_n{N_SYNTH}_seed{SEEDS[0]}_noweight"]
    if len(old_nw):
        refs["old_noweight_seed42"] = old_nw.assign(model_label="Tanpa class weight, 1 seed (Tabel 15, results/)")
    print("Referensi lama:", {k: sorted(v["seed"].tolist()) for k, v in refs.items()})
    return refs


def v2_group(prefix_kind, use_class_weight, stage):
    df = pd.DataFrame(ALL_ROWS)
    if not len(df) or "status" not in df.columns:
        return pd.DataFrame()
    df = df[df["status"].eq("ok")]
    m = (df["synth_kind"] == prefix_kind) & (_boolish(df["use_class_weight"]) == bool(use_class_weight)) & \
        (df["stage"].astype(int) == stage) & df["seed"].isin(SEEDS)
    return df[m].drop_duplicates("seed")


def summarize(groups):
    rows = []
    for label, g in groups:
        if g is None or not len(g):
            continue
        rec = {"Model": label, "n_seeds": int(g["seed"].nunique()),
               "seeds": ",".join(map(str, sorted(g["seed"].astype(int).unique())))}
        for m in METRICS:
            if m in g.columns:
                v = g[m].astype(float)
                rec[f"{METRIC_LABEL[m]} mean"] = float(v.mean())
                rec[f"{METRIC_LABEL[m]} std"] = float(v.std(ddof=1)) if len(v) > 1 else 0.0
                rec[f"{METRIC_LABEL[m]} (mean ± std)"] = (f"{v.mean():.4f} ± {v.std(ddof=1):.4f}"
                                                         if len(v) > 1 else f"{v.mean():.4f}")
        rows.append(rec)
    return pd.DataFrame(rows)


def paired_test(var_df, ref_df, metric, var_label, ref_label):
    a = ref_df.drop_duplicates("seed").set_index("seed")
    b = var_df.drop_duplicates("seed").set_index("seed")
    common = sorted(set(a.index) & set(b.index))
    rec = {"variant": var_label, "reference": ref_label, "metric": METRIC_LABEL.get(metric, metric),
           "n_pairs": len(common), "seeds": ",".join(map(str, common))}
    if len(common) < 2 or metric not in a.columns or metric not in b.columns:
        rec["status"] = "dilewati: pasangan seed < 2"
        return rec
    x = b.loc[common, metric].astype(float).to_numpy()
    r = a.loc[common, metric].astype(float).to_numpy()
    rec.update({"variant_mean": float(x.mean()), "reference_mean": float(r.mean()),
                "mean_diff_variant_minus_ref": float((x - r).mean()),
                "n_variant_better": int((x > r).sum())})
    try:
        if np.allclose(x, r, rtol=0, atol=1e-12):
            raise ValueError("semua selisih nol — uji t tidak terdefinisi")
        t, p = scipy_stats.ttest_rel(x, r)
        rec["t_stat"], rec["t_pvalue"] = float(t), float(p)
    except Exception as exc:
        rec["t_error"] = str(exc)
    try:
        if np.allclose(x - r, 0):
            raise ValueError("semua selisih nol")
        w, p = scipy_stats.wilcoxon(x, r, zero_method="wilcox", alternative="two-sided")
        rec["wilcoxon_stat"], rec["wilcoxon_pvalue"] = float(w), float(p)
    except Exception as exc:
        rec["wilcoxon_error"] = str(exc)
    rec["status"] = "ok"
    return rec


def holm_adjust(pvals):
    """Koreksi Holm–Bonferroni (step-down); NaN dibiarkan NaN."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    idx = np.where(~np.isnan(p))[0]
    m = len(idx)
    if m == 0:
        return out
    order = idx[np.argsort(p[idx], kind="stable")]
    running = 0.0
    for rank, i in enumerate(order):
        val = min(1.0, (m - rank) * p[i])
        running = max(running, val)
        out[i] = running
    return out


def run_paired_family(pairs, metrics=TEST_METRICS):
    """pairs: list (var_df, ref_df, var_label, ref_label). Holm dihitung atas SELURUH keluarga uji
    (semua pasangan × metrik), terpisah untuk uji t dan Wilcoxon."""
    rows = []
    for var_df, ref_df, vl, rl in pairs:
        if var_df is None or ref_df is None or not len(var_df) or not len(ref_df):
            continue
        for m in metrics:
            rows.append(paired_test(var_df, ref_df, m, vl, rl))
    df = pd.DataFrame(rows)
    if len(df):
        for col in ["t_pvalue", "wilcoxon_pvalue"]:
            if col not in df.columns:
                df[col] = np.nan
        df["t_pvalue_holm"] = holm_adjust(df["t_pvalue"])
        df["wilcoxon_pvalue_holm"] = holm_adjust(df["wilcoxon_pvalue"])
        df["family_size"] = int(df["t_pvalue"].notna().sum())
    return df


def to_markdown(df, floatfmt="{:.4f}"):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in df.iterrows():
        vals = []
        for v in row.tolist():
            if isinstance(v, float):
                vals.append("" if np.isnan(v) else floatfmt.format(v))
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def save_table(df, name, title=None):
    df.to_csv(guard_write(TABLE_DIR / f"{name}.csv"), index=False)
    md_txt = (f"### {title}\n\n" if title else "") + to_markdown(df) + "\n"
    guard_write(TABLE_DIR / f"{name}.md").write_text(md_txt)
    print(f"Tersimpan: tables/{name}.csv dan .md")


def compact_summary(df):
    keep = ["Model", "n_seeds"] + [f"{METRIC_LABEL[m]} (mean ± std)" for m in
                                   ["f1_macro", "Web Attack__precision", "Web Attack__recall", "Web Attack__f1"]]
    return df[[c for c in keep if c in df.columns]]


def compact_tests(df):
    keep = ["variant", "reference", "metric", "n_pairs", "variant_mean", "reference_mean",
            "mean_diff_variant_minus_ref", "n_variant_better", "t_pvalue", "t_pvalue_holm",
            "wilcoxon_pvalue", "wilcoxon_pvalue_holm"]
    return df[[c for c in keep if c in df.columns]]


def comparison_lines(tests_df, variant, reference, alpha=0.05):
    """Kalimat per metrik untuk satu perbandingan (varian vs referensi), memakai p uji t berpasangan
    yang sudah dikoreksi Holm. Mengembalikan (daftar kalimat, dict status per metrik).
    Status: 'sig_lebih_tinggi', 'sig_lebih_rendah', 'rata2_lebih_tinggi', 'rata2_lebih_rendah', 'sama', 'n/a'."""
    lines, status = [], {}
    if tests_df is None or not len(tests_df):
        return [f"  (uji {variant} vs {reference} tidak tersedia)"], status
    sub = tests_df[(tests_df["variant"] == variant) & (tests_df["reference"] == reference)]
    for _, r in sub.iterrows():
        m = r["metric"]
        d = r.get("mean_diff_variant_minus_ref", np.nan)
        p_h = r.get("t_pvalue_holm", np.nan)
        p_w = r.get("wilcoxon_pvalue", np.nan)
        nb = r.get("n_variant_better", np.nan)
        npairs = r.get("n_pairs", np.nan)
        if pd.isna(d):
            status[m] = "n/a"
            lines.append(f"  - {m}: tidak dapat diuji ({r.get('status', '')})")
            continue
        if abs(d) < 1e-9:
            st = "sama"
        elif not pd.isna(p_h) and p_h < alpha:
            st = "sig_lebih_tinggi" if d > 0 else "sig_lebih_rendah"
        else:
            st = "rata2_lebih_tinggi" if d > 0 else "rata2_lebih_rendah"
        status[m] = st
        txt = {"sig_lebih_tinggi": "LEBIH TINGGI dan SIGNIFIKAN",
               "sig_lebih_rendah": "LEBIH RENDAH dan SIGNIFIKAN",
               "rata2_lebih_tinggi": "lebih tinggi rata-rata saja (tidak signifikan setelah Holm)",
               "rata2_lebih_rendah": "lebih rendah rata-rata (tidak signifikan setelah Holm)",
               "sama": "sama persis"}[st]
        lines.append(f"  - {m}: {r['variant_mean']:.4f} vs {r['reference_mean']:.4f} (selisih {d:+.4f}; "
                     f"lebih baik di {int(nb)}/{int(npairs)} seed; p-t Holm={p_h:.4f}; p-Wilcoxon={p_w:.4f}) → {txt}")
    return lines, status


OLD_REFS = load_old_reference()

Referensi lama: {'main_ctgan': [42, 123, 456, 789, 1024], 'smote': [42, 123, 456, 789, 1024], 'weights_only': [42, 123, 456, 789, 1024], 'undersample_only': [42, 123, 456, 789, 1024], 'old_noweight_seed42': [42]}


In [11]:
# ============================================================
# Ambang Web Attack (VEKTORISASI, tanpa loop per ambang di seluruh test) + filter LightGBM
# ============================================================
def threshold_metrics_vectorized(proba, y_true, classes, thresholds):
    """Metrik multikelas untuk banyak ambang sekaligus.
    Prediksi(t) = Web Attack jika p_web >= t, selain itu argmax kelas lain (sama dengan notebook sebelumnya).
    Matriks konfusi semua ambang dihitung dari satu kali pengurutan + cumsum: O(N log N), bukan O(N·T)."""
    classes = list(classes)
    K = len(classes)
    w = classes.index("Web Attack")
    y_true = np.asarray(y_true, dtype=np.int64)
    p = proba[:, w].astype(np.float64)
    alt = proba.astype(np.float64).copy()
    alt[:, w] = -np.inf
    alt_pred = alt.argmax(axis=1)
    cm_alt = np.bincount(y_true * K + alt_pred, minlength=K * K).astype(np.int64)
    thr = np.asarray(thresholds, dtype=np.float64)
    cand = np.where(p >= thr.min())[0]                       # hanya baris yang bisa berpindah ke Web
    order = cand[np.argsort(-p[cand], kind="stable")]
    ps = p[order]
    M = len(order)
    delta = np.zeros((M + 1, K * K), dtype=np.int32)
    rows_ = np.arange(1, M + 1)
    delta[rows_, y_true[order] * K + w] += 1
    delta[rows_, y_true[order] * K + alt_pred[order]] -= 1
    cum = np.cumsum(delta, axis=0)
    k = np.searchsorted(-ps, -thr, side="right")             # jumlah baris dengan p >= t
    cms = (cm_alt[None, :] + cum[k]).reshape(len(thr), K, K)  # (T, K, K): baris=aktual, kolom=prediksi
    tp = np.einsum("tkk->tk", cms).astype(float)
    pred_sum = cms.sum(axis=1).astype(float)
    true_sum = cms.sum(axis=2).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(pred_sum > 0, tp / pred_sum, 0.0)
        rec = np.where(true_sum > 0, tp / true_sum, 0.0)
        f1 = np.where(prec + rec > 0, 2 * prec * rec / (prec + rec), 0.0)
    present = (true_sum + pred_sum) > 0
    f1_macro = (f1 * present).sum(axis=1) / present.sum(axis=1)
    out = pd.DataFrame({"threshold": thr, "accuracy": tp.sum(axis=1) / len(y_true), "f1_macro": f1_macro,
                        "Web Attack__precision": prec[:, w], "Web Attack__recall": rec[:, w],
                        "Web Attack__f1": f1[:, w], "n_web_pred": pred_sum[:, w].astype(int)})
    return out, cms


def threshold_analysis(proba, y_true, classes, tag):
    classes = list(classes)
    w = classes.index("Web Attack")
    y_bin = (np.asarray(y_true) == w).astype(int)
    p_web = proba[:, w]
    prec_c, rec_c, thr_c = precision_recall_curve(y_bin, p_web)
    ap = float(average_precision_score(y_bin, p_web))
    if len(thr_c):
        f1_bins = 2 * prec_c[:-1] * rec_c[:-1] / np.clip(prec_c[:-1] + rec_c[:-1], 1e-12, None)
        bi = int(np.nanargmax(f1_bins))
        best_t, best_f1 = float(thr_c[bi]), float(f1_bins[bi])
    else:
        best_t, best_f1 = 0.5, np.nan
    tbl, _ = threshold_metrics_vectorized(proba, y_true, classes, list(THRESHOLDS) + [best_t])
    tbl["is_best_binary_f1"] = np.isclose(tbl["threshold"], best_t)
    tbl["average_precision"] = ap
    tbl = tbl.drop_duplicates(subset=["threshold"]).reset_index(drop=True)
    # Pemeriksaan silang SATU ambang (0.5) dengan sklearn — satu kali, bukan per ambang
    alt05 = proba.copy()
    alt05[:, w] = -np.inf
    pred05 = np.where(p_web >= 0.5, w, alt05.argmax(axis=1))
    chk, _ = metrics_from_preds(np.asarray(y_true), pred05, classes)
    r05 = tbl.loc[np.isclose(tbl["threshold"], 0.5)].iloc[0]
    ok = abs(chk["f1_macro"] - r05["f1_macro"]) < 1e-9 and abs(chk["Web Attack__f1"] - r05["Web Attack__f1"]) < 1e-9
    print(f"[{tag}] AP={ap:.4f} | ambang F1-biner terbaik={best_t:.4f} (F1={best_f1:.4f}) | cek silang sklearn: {'OK' if ok else 'BEDA!'}")
    return tbl, ap, best_t, (prec_c, rec_c)


_SECONDARY_FILTER_CACHE = {}


def get_secondary_filter():
    """Filter dilatih SEKALI per sesi lalu dipakai ulang oleh Tahap 1 dan Tahap 1b (setelan identik)."""
    if "pred_test" not in _SECONDARY_FILTER_CACHE:
        clf, name = train_secondary_filter()
        _SECONDARY_FILTER_CACHE.update(name=name, pred_test=clf.predict(X_test_mm))
    return _SECONDARY_FILTER_CACHE["pred_test"], _SECONDARY_FILTER_CACHE["name"]


def train_secondary_filter():
    """Filter biner Web vs lainnya pada TRAIN asli (MinMax), sama dengan notebook sebelumnya."""
    y_bin_tr = (y_train.values == "Web Attack").astype(int)
    if LGBM_AVAILABLE:
        clf = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=SPLIT_SEED,
                             class_weight="balanced", n_jobs=-1, verbose=-1)
        name = "lightgbm"
    else:
        clf = RandomForestClassifier(n_estimators=200, random_state=SPLIT_SEED, class_weight="balanced", n_jobs=-1)
        name = "random_forest"
    t0 = time.perf_counter()
    clf.fit(X_train_mm, y_bin_tr)
    print(f"Filter {name} dilatih {time.perf_counter() - t0:.1f}s pada {len(X_train_mm):,} baris train.")
    return clf, name


def apply_secondary_filter(proba, y_true, classes, filter_pred_test):
    classes = list(classes)
    w = classes.index("Web Attack")
    base = proba.argmax(axis=1)
    alt = proba.copy()
    alt[:, w] = -np.inf
    second = alt.argmax(axis=1)
    mask = (base == w) & (filter_pred_test == 0)
    new = base.copy()
    new[mask] = second[mask]
    r0, _ = metrics_from_preds(y_true, base, classes)
    r1, _ = metrics_from_preds(y_true, new, classes)
    return r0, r1, base, new, int(mask.sum())

## Tahap 1 — CTGAN asli + CNN-LSTM **tanpa class weight**, 5 seed

Data sintetis CTGAN per seed dibaca dari `results/X_web_synth_n5000_seed{seed}_n5000.csv` (cache model utama). Dengan begitu, data latih setiap seed **identik** dengan model utama di Tabel 16; satu-satunya perbedaan adalah class weight dimatikan. Jika cache suatu seed tidak ada, CTGAN dilatih ulang dengan setelan yang sama dan disimpan di `results_v2/synth_cache/`.

**Di v3:** kelima run Tahap 1 sudah selesai (dijalankan dengan notebook v2) dan tercatat di `results_v2/all_runs_v2.csv`, jadi sel pelatihan di bawah hanya mencetak `SKIP`. Tabel, analisis ambang, dan filter dihitung ulang dari `proba_*.npz` yang sudah ada (beberapa menit saja).

Catatan: hasil seed 42 di sini bisa sedikit berbeda dari baris "Without class weights" di Tabel 15 (0.883), karena run lama itu memakai fit CTGAN tersendiri (cache `n5000_noweight`), bukan data sintetis model utama.

In [12]:
# ============================================================
# TAHAP 1 — pelatihan (5 run, resume otomatis)
# ============================================================
STAGE1_JOBS = [make_job(1, "original", False, s) for s in SEEDS]
if RUN_STAGE1:
    run_jobs(STAGE1_JOBS, "Tahap 1")
else:
    print("Tahap 1 dilewati (RUN_STAGE1=False).")

Tahap 1: 5 run terencana, sudah selesai 5, sisa 0.

[Tahap 1 — run 1 of 5] RUN s1_main_ctgan_cnn_lstm_w0_n5000_seed42
[Tahap 1 — run 1 of 5] SKIP — sudah status=ok di all_runs_v2.csv
Waktu berjalan Tahap 1: 0.0 menit

[Tahap 1 — run 2 of 5] RUN s1_main_ctgan_cnn_lstm_w0_n5000_seed123
[Tahap 1 — run 2 of 5] SKIP — sudah status=ok di all_runs_v2.csv
Waktu berjalan Tahap 1: 0.0 menit

[Tahap 1 — run 3 of 5] RUN s1_main_ctgan_cnn_lstm_w0_n5000_seed456
[Tahap 1 — run 3 of 5] SKIP — sudah status=ok di all_runs_v2.csv
Waktu berjalan Tahap 1: 0.0 menit

[Tahap 1 — run 4 of 5] RUN s1_main_ctgan_cnn_lstm_w0_n5000_seed789
[Tahap 1 — run 4 of 5] SKIP — sudah status=ok di all_runs_v2.csv
Waktu berjalan Tahap 1: 0.0 menit

[Tahap 1 — run 5 of 5] RUN s1_main_ctgan_cnn_lstm_w0_n5000_seed1024
[Tahap 1 — run 5 of 5] SKIP — sudah status=ok di all_runs_v2.csv
Waktu berjalan Tahap 1: 0.0 menit


In [13]:
# ============================================================
# TAHAP 1 — ringkasan mean ± std, uji berpasangan, koreksi Holm
# ============================================================
S1_NOWEIGHT = v2_group("original", False, 1)
S1_LABEL = "Targeted CTGAN, no class weights (v2, Tahap 1)"
STAGE1_SUMMARY = pd.DataFrame()
STAGE1_TESTS = pd.DataFrame()
if len(S1_NOWEIGHT):
    groups = [(S1_LABEL, S1_NOWEIGHT)]
    for key in ["main_ctgan", "smote", "weights_only", "undersample_only", "old_noweight_seed42"]:
        if key in OLD_REFS:
            groups.append((OLD_REFS[key]["model_label"].iloc[0], OLD_REFS[key]))
    STAGE1_SUMMARY = summarize(groups)
    save_table(STAGE1_SUMMARY, "stage1_summary_full", "Tahap 1 — mean ± std (lengkap)")
    save_table(compact_summary(STAGE1_SUMMARY), "stage1_summary", "Tahap 1 — mean ± std (5 seed)")
    display(compact_summary(STAGE1_SUMMARY))

    pairs = []
    if "main_ctgan" in OLD_REFS:
        pairs.append((S1_NOWEIGHT, OLD_REFS["main_ctgan"], "CTGAN no-weight (v2)", "CTGAN + weights (main)"))
    if "smote" in OLD_REFS:
        pairs.append((S1_NOWEIGHT, OLD_REFS["smote"], "CTGAN no-weight (v2)", "SMOTE + weights"))
    STAGE1_TESTS = run_paired_family(pairs)
    if len(STAGE1_TESTS):
        save_table(STAGE1_TESTS, "stage1_paired_tests_full", "Tahap 1 — uji berpasangan (lengkap)")
        save_table(compact_tests(STAGE1_TESTS), "stage1_paired_tests",
                   "Tahap 1 — uji t berpasangan & Wilcoxon, koreksi Holm atas seluruh keluarga uji")
        display(compact_tests(STAGE1_TESTS))
        print("Catatan: dengan 5 pasangan, p Wilcoxon dua sisi terkecil yang mungkin adalah 0.0625. "
              "Koreksi Holm dihitung atas semua pasangan × metrik di Tahap 1 (terpisah untuk uji t dan Wilcoxon).")
    else:
        print("Uji berpasangan dilewati: referensi results/all_runs.csv tidak ditemukan.")
    if len(S1_NOWEIGHT) < len(SEEDS):
        print(f"PERINGATAN: baru {len(S1_NOWEIGHT)} dari {len(SEEDS)} seed selesai — jalankan lagi sel pelatihan.")
else:
    print("Belum ada run Tahap 1 berstatus ok.")

Tersimpan: tables/stage1_summary_full.csv dan .md
Tersimpan: tables/stage1_summary.csv dan .md


,Model,n_seeds,Macro F1 (mean ± std),WA precision (mean ± std),WA recall (mean ± std),WA F1 (mean ± std)
0,"Targeted CTGAN, no class weights (v2, Tahap 1)",5,0.8655 ± 0.0205,0.9164 ± 0.0795,0.7444 ± 0.2366,0.7970 ± 0.1153
1,"Targeted CTGAN + class weights (Tabel 16, resu...",5,0.8176 ± 0.0334,0.3973 ± 0.1900,1.0000 ± 0.0000,0.5465 ± 0.2050
2,SMOTE + class weights (results/),5,0.7921 ± 0.0082,0.2450 ± 0.0684,1.0000 ± 0.0000,0.3899 ± 0.0849
3,"No augmentation, class weights only (results/)",5,0.7288 ± 0.0476,0.1282 ± 0.2680,0.8944 ± 0.2360,0.1195 ± 0.2312
4,Undersample only (results/),5,0.7239 ± 0.0422,0.0886 ± 0.0971,0.8917 ± 0.2346,0.1429 ± 0.1527
5,"Tanpa class weight, 1 seed (Tabel 15, results/)",1,0.8829,0.8161,0.9861,0.8931


Tersimpan: tables/stage1_paired_tests_full.csv dan .md
Tersimpan: tables/stage1_paired_tests.csv dan .md


,variant,reference,metric,n_pairs,variant_mean,reference_mean,mean_diff_variant_minus_ref,n_variant_better,t_pvalue,t_pvalue_holm,wilcoxon_pvalue,wilcoxon_pvalue_holm
0,CTGAN no-weight (v2),CTGAN + weights (main),Macro F1,5,0.8655,0.8176,0.0479,5,0.0444,0.1776,0.0625,0.5000
1,CTGAN no-weight (v2),CTGAN + weights (main),WA precision,5,0.9164,0.3973,0.5191,5,0.0062,0.0310,0.0625,0.5000
2,CTGAN no-weight (v2),CTGAN + weights (main),WA recall,5,0.7444,1.0000,-0.2556,0,0.0732,0.2195,0.2500,0.5000
3,CTGAN no-weight (v2),CTGAN + weights (main),WA F1,5,0.7970,0.5465,0.2505,4,0.0749,0.2195,0.1250,0.5000
4,CTGAN no-weight (v2),SMOTE + weights,Macro F1,5,0.8655,0.7921,0.0733,5,0.0022,0.0151,0.0625,0.5000
5,CTGAN no-weight (v2),SMOTE + weights,WA precision,5,0.9164,0.2450,0.6714,5,0.0001,0.0006,0.0625,0.5000
6,CTGAN no-weight (v2),SMOTE + weights,WA recall,5,0.7444,1.0000,-0.2556,0,0.0732,0.2195,0.2500,0.5000
7,CTGAN no-weight (v2),SMOTE + weights,WA F1,5,0.7970,0.3899,0.4071,5,0.0041,0.0243,0.0625,0.5000


Catatan: dengan 5 pasangan, p Wilcoxon dua sisi terkecil yang mungkin adalah 0.0625. Koreksi Holm dihitung atas semua pasangan × metrik di Tahap 1 (terpisah untuk uji t dan Wilcoxon).


In [14]:
# ============================================================
# TAHAP 1 — analisis ambang (vektorisasi) + filter LightGBM tahap kedua
# Filter dilatih SEKALI per sesi (± 20 detik di Colab), lalu dipakai Tahap 1 dan Tahap 1b.
# ============================================================
def load_proba(run_name):
    path = RUN_DIR / f"proba_{run_name}.npz"
    if not path.exists():
        return None
    d = np.load(path, allow_pickle=True)
    return d["proba"], d["y_true"], [str(c) for c in d["classes"]]


S1_FILTER_TABLE = pd.DataFrame()
S1_THRESHOLD_TABLE = pd.DataFrame()
_s1_names = [j["run_name"] for j in STAGE1_JOBS if find_ok(j["run_name"]) is not None]
if _s1_names:
    seed0_name = make_job(1, "original", False, SEEDS[0])["run_name"]
    # Ambang + kurva PR untuk seed pertama (42)
    loaded = load_proba(seed0_name)
    if loaded is not None and "Web Attack" in loaded[2]:
        proba, yt, classes = loaded
        S1_THRESHOLD_TABLE, ap, best_t, (pc, rc) = threshold_analysis(proba, yt, classes, seed0_name)
        save_table(S1_THRESHOLD_TABLE, "stage1_threshold_web_attack_seed42",
                   "Tahap 1 — ambang Web Attack, model tanpa class weight (seed 42)")
        display(S1_THRESHOLD_TABLE)
        plt.figure(figsize=(6, 5))
        plt.plot(rc, pc, label=f"Web Attack AP={ap:.4f}")
        plt.xlabel("Recall"); plt.ylabel("Precision")
        plt.title("Precision–Recall — Web Attack (no class weights, seed 42)")
        plt.legend(); plt.grid(True, linestyle=":", alpha=0.7); plt.tight_layout()
        plt.savefig(guard_write(FIG_DIR / "stage1_pr_curve_web_attack_seed42.png"), dpi=300, bbox_inches="tight")
        plt.close()

    # Filter tahap kedua pada semua seed Tahap 1
    filt_pred_test, filt_name = get_secondary_filter()
    rows = []
    for name in _s1_names:
        loaded = load_proba(name)
        if loaded is None:
            continue
        proba, yt, classes = loaded
        r0, r1, base, new, n_rr = apply_secondary_filter(proba, yt, classes, filt_pred_test)
        seed = int(name.rsplit("seed", 1)[1])
        for pipe, r in [("cnn_lstm_only", r0), (f"cnn_lstm_then_{filt_name}", r1)]:
            rows.append({"seed": seed, "pipeline": pipe, "accuracy": r["accuracy"], "f1_macro": r["f1_macro"],
                         "Web Attack__precision": r["Web Attack__precision"], "Web Attack__recall": r["Web Attack__recall"],
                         "Web Attack__f1": r["Web Attack__f1"], "n_rerouted": n_rr if pipe != "cnn_lstm_only" else 0})
        if seed == SEEDS[0]:
            save_confusion(yt, new, classes, FIG_DIR / f"stage1_cm_seed42_then_{filt_name}",
                           f"CM — no class weights + {filt_name} filter (seed 42)")
    S1_FILTER_TABLE = pd.DataFrame(rows)
    save_table(S1_FILTER_TABLE, "stage1_secondary_filter_per_seed", "Tahap 1 — filter tahap kedua per seed")
    agg = (S1_FILTER_TABLE.groupby("pipeline")[["f1_macro", "Web Attack__precision", "Web Attack__recall", "Web Attack__f1"]]
           .agg(["mean", "std"]))
    agg.columns = [f"{a} {b}" for a, b in agg.columns]
    agg = agg.reset_index()
    save_table(agg, "stage1_secondary_filter_mean_std", "Tahap 1 — filter tahap kedua, mean ± std antar seed")
    display(S1_FILTER_TABLE[S1_FILTER_TABLE["seed"] == SEEDS[0]])
    display(agg)
    # Gambar CM model tanpa class weight (seed 42) untuk naskah
    cm_csv = RUN_DIR / f"cm_{seed0_name}.csv"
    if cm_csv.exists():
        shutil.copy2(cm_csv.with_suffix(".png"), guard_write(FIG_DIR / "stage1_cm_noweight_seed42.png"))
else:
    print("Analisis ambang/filter Tahap 1 dilewati (belum ada run ok).")

[s1_main_ctgan_cnn_lstm_w0_n5000_seed42] AP=0.9530 | ambang F1-biner terbaik=0.4770 (F1=0.9536) | cek silang sklearn: OK
Tersimpan: tables/stage1_threshold_web_attack_seed42.csv dan .md


,threshold,accuracy,f1_macro,Web Attack__precision,Web Attack__recall,Web Attack__f1,n_web_pred,is_best_binary_f1,average_precision
0,0.5000,0.9710,0.8591,0.9375,0.6250,0.7500,48,False,0.9530
1,0.5500,0.9710,0.8566,0.9556,0.5972,0.7350,45,False,0.9530
2,0.6000,0.9710,0.8566,0.9556,0.5972,0.7350,45,False,0.9530
3,0.6500,0.9710,0.8548,0.9545,0.5833,0.7241,44,False,0.9530
4,0.7000,0.9710,0.8548,0.9545,0.5833,0.7241,44,False,0.9530
5,0.7500,0.9710,0.8548,0.9545,0.5833,0.7241,44,False,0.9530
6,0.8000,0.9710,0.8548,0.9545,0.5833,0.7241,44,False,0.9530
7,0.8500,0.9710,0.8529,0.9535,0.5694,0.7130,43,False,0.9530
8,0.9000,0.9710,0.8529,0.9535,0.5694,0.7130,43,False,0.9530
9,0.4770,0.9710,0.8930,0.9114,1.0000,0.9536,79,True,0.9530


Filter lightgbm dilatih 18.0s pada 2,509,032 baris train.
Tersimpan: tables/stage1_secondary_filter_per_seed.csv dan .md
Tersimpan: tables/stage1_secondary_filter_mean_std.csv dan .md


,seed,pipeline,accuracy,f1_macro,Web Attack__precision,Web Attack__recall,Web Attack__f1,n_rerouted
0,42,cnn_lstm_only,0.9710,0.8591,0.9375,0.6250,0.7500,0
1,42,cnn_lstm_then_lightgbm,0.9710,0.8623,1.0000,0.6250,0.7692,3


,pipeline,f1_macro mean,f1_macro std,Web Attack__precision mean,Web Attack__precision std,Web Attack__recall mean,Web Attack__recall std,Web Attack__f1 mean,Web Attack__f1 std
0,cnn_lstm_only,0.8655,0.0205,0.9164,0.0795,0.7444,0.2366,0.7970,0.1153
1,cnn_lstm_then_lightgbm,0.8712,0.0255,0.9892,0.0148,0.7444,0.2366,0.8315,0.1450


In [15]:
# ============================================================
# TAHAP 1 — kesimpulan PER PERBANDINGAN (menggantikan label tunggal KUAT/SEDANG/LEMAH di v2)
# ============================================================
STAGE1_VERDICT_LINES = []
if len(STAGE1_TESTS):
    for ref in ["CTGAN + weights (main)", "SMOTE + weights"]:
        lines, _ = comparison_lines(STAGE1_TESTS, "CTGAN no-weight (v2)", ref)
        STAGE1_VERDICT_LINES.append(f"CTGAN tanpa class weight vs {ref}:")
        STAGE1_VERDICT_LINES.extend(lines)
    STAGE1_VERDICT_LINES.append("Catatan: pembanding SMOTE di atas MEMAKAI class weight, jadi selisihnya bisa berasal dari "
                                "class weight, bukan dari CTGAN. Perbandingan yang adil ada di Tahap 1b.")
    if len(S1_NOWEIGHT) < len(SEEDS):
        STAGE1_VERDICT_LINES.append(f"PERINGATAN: baru {len(S1_NOWEIGHT)} dari {len(SEEDS)} seed Tahap 1 selesai.")
    print("KESIMPULAN TAHAP 1 (p = uji t berpasangan, koreksi Holm atas seluruh keluarga uji Tahap 1):")
    print("\n".join(STAGE1_VERDICT_LINES))
    guard_write(TABLE_DIR / "stage1_verdict.txt").write_text("\n".join(STAGE1_VERDICT_LINES) + "\n")
else:
    print("Kesimpulan Tahap 1 dilewati (belum ada uji berpasangan).")

KESIMPULAN TAHAP 1 (p = uji t berpasangan, koreksi Holm atas seluruh keluarga uji Tahap 1):
CTGAN tanpa class weight vs CTGAN + weights (main):
  - Macro F1: 0.8655 vs 0.8176 (selisih +0.0479; lebih baik di 5/5 seed; p-t Holm=0.1776; p-Wilcoxon=0.0625) → lebih tinggi rata-rata saja (tidak signifikan setelah Holm)
  - WA precision: 0.9164 vs 0.3973 (selisih +0.5191; lebih baik di 5/5 seed; p-t Holm=0.0310; p-Wilcoxon=0.0625) → LEBIH TINGGI dan SIGNIFIKAN
  - WA recall: 0.7444 vs 1.0000 (selisih -0.2556; lebih baik di 0/5 seed; p-t Holm=0.2195; p-Wilcoxon=0.2500) → lebih rendah rata-rata (tidak signifikan setelah Holm)
  - WA F1: 0.7970 vs 0.5465 (selisih +0.2505; lebih baik di 4/5 seed; p-t Holm=0.2195; p-Wilcoxon=0.1250) → lebih tinggi rata-rata saja (tidak signifikan setelah Holm)
CTGAN tanpa class weight vs SMOTE + weights:
  - Macro F1: 0.8655 vs 0.7921 (selisih +0.0733; lebih baik di 5/5 seed; p-t Holm=0.0151; p-Wilcoxon=0.0625) → LEBIH TINGGI dan SIGNIFIKAN
  - WA precision: 0.916

## Tahap 1b — SMOTE + CNN-LSTM **tanpa class weight**, 5 seed (BARU di v3)

Tujuan: menghapus faktor perancu. Di Tahap 1, CTGAN **tanpa** class weight dibandingkan dengan SMOTE **dengan** class weight, sehingga kenaikannya bisa berasal dari dimatikannya class weight, bukan dari CTGAN.

Data latih Tahap 1b **identik** dengan baseline `smote_cnn_lstm_w1_n5000_seed{seed}` di `results/`: Web Attack train di-oversample dengan SMOTE (`k_neighbors=5`, `random_state=seed`) dari 290 menjadi 290 + 5000 = 5290 baris, kelas lain tidak diubah, lalu Benign di-undersample (seed yang sama) sebanyak jumlah baris serangan. Arsitektur, 50 epoch, patience 5, batch 512, float32, dan test (tidak diubah) sama persis. **Satu-satunya beda: class weight dimatikan.**

Perbandingan utama: **CTGAN tanpa class weight (Tahap 1) vs SMOTE tanpa class weight (Tahap 1b)** — keduanya tanpa class weight, jadi bedanya hanya metode augmentasi. Perbandingan tambahan: SMOTE tanpa vs dengan class weight (efek class weight pada SMOTE).

In [16]:
# ============================================================
# TAHAP 1b — pelatihan (5 run, resume otomatis)
# ============================================================
STAGE1B_JOBS = [make_job(1, "smote", False, s) for s in SEEDS]
if RUN_STAGE1B:
    run_jobs(STAGE1B_JOBS, "Tahap 1b")
else:
    print("Tahap 1b dilewati (RUN_STAGE1B=False).")

Tahap 1b: 5 run terencana, sudah selesai 0, sisa 5.

[Tahap 1b — run 1 of 5] RUN s1b_smote_cnn_lstm_w0_n5000_seed42
smote: Web Attack 290 -> 5290 (target 5290, k=5, seed=42)
Distribusi train (smote):
 Label
Benign                      791486
DoS attacks-Hulk            369530
FTP-BruteForce              154683
SSH-Bruteforce              150071
DoS attacks-SlowHTTPTest    111912
Web Attack                    5290
Name: count, dtype: int64
class_weight: TIDAK dipakai
Epoch 1/50
2474/2474 - 31s - 12ms/step - accuracy: 0.8983 - loss: 0.2288 - val_accuracy: 0.9483 - val_loss: 0.1010
Epoch 2/50
2474/2474 - 25s - 10ms/step - accuracy: 0.9440 - loss: 0.1096 - val_accuracy: 0.9488 - val_loss: 0.0983
Epoch 3/50
2474/2474 - 25s - 10ms/step - accuracy: 0.9466 - loss: 0.1013 - val_accuracy: 0.9483 - val_loss: 0.0981
Epoch 4/50
2474/2474 - 25s - 10ms/step - accuracy: 0.9471 - loss: 0.1002 - val_accuracy: 0.9489 - val_loss: 0.0976
Epoch 5/50
2474/2474 - 25s - 10ms/step - accuracy: 0.9475 - loss: 0.0

In [17]:
# ============================================================
# TAHAP 1b — ringkasan mean ± std, uji berpasangan, koreksi Holm
# ============================================================
S1B_SMOTE_NW = v2_group("smote", False, 1)
S1B_LABEL = "SMOTE, no class weights (v3, Tahap 1b)"
KEY_VAR, KEY_REF = "CTGAN no-weight (v2)", "SMOTE no-weight (v3)"
CW_VAR, CW_REF = "SMOTE no-weight (v3)", "SMOTE + weights"
STAGE1B_SUMMARY = pd.DataFrame()
STAGE1B_TESTS = pd.DataFrame()
if len(S1B_SMOTE_NW):
    groups = []
    if len(S1_NOWEIGHT):
        groups.append((S1_LABEL, S1_NOWEIGHT))
    groups.append((S1B_LABEL, S1B_SMOTE_NW))
    for key in ["main_ctgan", "smote"]:
        if key in OLD_REFS:
            groups.append((OLD_REFS[key]["model_label"].iloc[0], OLD_REFS[key]))
    STAGE1B_SUMMARY = summarize(groups)
    save_table(STAGE1B_SUMMARY, "stage1b_summary_full", "Tahap 1b — mean ± std (lengkap)")
    save_table(compact_summary(STAGE1B_SUMMARY), "stage1b_summary",
               "Tahap 1b — mean ± std (5 seed): CTGAN vs SMOTE, dengan dan tanpa class weight")
    display(compact_summary(STAGE1B_SUMMARY))

    pairs = []
    if len(S1_NOWEIGHT):
        pairs.append((S1_NOWEIGHT, S1B_SMOTE_NW, KEY_VAR, KEY_REF))          # perbandingan utama
    if "smote" in OLD_REFS:
        pairs.append((S1B_SMOTE_NW, OLD_REFS["smote"], CW_VAR, CW_REF))      # efek class weight pada SMOTE
    STAGE1B_TESTS = run_paired_family(pairs)
    if len(STAGE1B_TESTS):
        save_table(STAGE1B_TESTS, "stage1b_paired_tests_full", "Tahap 1b — uji berpasangan (lengkap)")
        save_table(compact_tests(STAGE1B_TESTS), "stage1b_paired_tests",
                   "Tahap 1b — uji t berpasangan & Wilcoxon, koreksi Holm atas seluruh keluarga uji Tahap 1b "
                   "(2 perbandingan × 4 metrik)")
        display(compact_tests(STAGE1B_TESTS))
        print("Catatan: Holm dihitung atas 2 perbandingan × 4 metrik = 8 uji (terpisah untuk uji t dan Wilcoxon). "
              "Dengan 5 pasangan, p Wilcoxon dua sisi terkecil yang mungkin adalah 0.0625.")
    if len(S1B_SMOTE_NW) < len(SEEDS):
        print(f"PERINGATAN: baru {len(S1B_SMOTE_NW)} dari {len(SEEDS)} seed Tahap 1b selesai — jalankan lagi (Run all).")
else:
    print("Belum ada run Tahap 1b berstatus ok.")

Tersimpan: tables/stage1b_summary_full.csv dan .md
Tersimpan: tables/stage1b_summary.csv dan .md


,Model,n_seeds,Macro F1 (mean ± std),WA precision (mean ± std),WA recall (mean ± std),WA F1 (mean ± std)
0,"Targeted CTGAN, no class weights (v2, Tahap 1)",5,0.8655 ± 0.0205,0.9164 ± 0.0795,0.7444 ± 0.2366,0.7970 ± 0.1153
1,"SMOTE, no class weights (v3, Tahap 1b)",5,0.8829 ± 0.0030,0.8135 ± 0.0293,1.0000 ± 0.0000,0.8969 ± 0.0179
2,"Targeted CTGAN + class weights (Tabel 16, resu...",5,0.8176 ± 0.0334,0.3973 ± 0.1900,1.0000 ± 0.0000,0.5465 ± 0.2050
3,SMOTE + class weights (results/),5,0.7921 ± 0.0082,0.2450 ± 0.0684,1.0000 ± 0.0000,0.3899 ± 0.0849


Tersimpan: tables/stage1b_paired_tests_full.csv dan .md
Tersimpan: tables/stage1b_paired_tests.csv dan .md


,variant,reference,metric,n_pairs,variant_mean,reference_mean,mean_diff_variant_minus_ref,n_variant_better,t_pvalue,t_pvalue_holm,wilcoxon_pvalue,wilcoxon_pvalue_holm
0,CTGAN no-weight (v2),SMOTE no-weight (v3),Macro F1,5,0.8655,0.8829,-0.0174,2,0.1558,0.3045,0.3125,0.7500
1,CTGAN no-weight (v2),SMOTE no-weight (v3),WA precision,5,0.9164,0.8135,0.1029,5,0.0184,0.0735,0.0625,0.4375
2,CTGAN no-weight (v2),SMOTE no-weight (v3),WA recall,5,0.7444,1.0000,-0.2556,0,0.0732,0.2195,0.2500,0.7500
3,CTGAN no-weight (v2),SMOTE no-weight (v3),WA F1,5,0.7970,0.8969,-0.1000,2,0.1523,0.3045,0.3125,0.7500
4,SMOTE no-weight (v3),SMOTE + weights,Macro F1,5,0.8829,0.7921,0.0907,5,0.0000,0.0001,0.0625,0.4375
5,SMOTE no-weight (v3),SMOTE + weights,WA precision,5,0.8135,0.2450,0.5685,5,0.0000,0.0002,0.0625,0.4375
6,SMOTE no-weight (v3),SMOTE + weights,WA recall,5,1.0000,1.0000,0.0000,0,NaN,NaN,NaN,NaN
7,SMOTE no-weight (v3),SMOTE + weights,WA F1,5,0.8969,0.3899,0.5070,5,0.0001,0.0006,0.0625,0.4375


Catatan: Holm dihitung atas 2 perbandingan × 4 metrik = 8 uji (terpisah untuk uji t dan Wilcoxon). Dengan 5 pasangan, p Wilcoxon dua sisi terkecil yang mungkin adalah 0.0625.


In [18]:
# ============================================================
# TAHAP 1b — filter LightGBM tahap kedua pada SMOTE tanpa class weight
# (filter dan cara penerapan IDENTIK dengan Tahap 1, sehingga pipeline ber-filter sebanding)
# ============================================================
S1B_FILTER_TABLE = pd.DataFrame()
S1B_FILTER_TESTS = pd.DataFrame()
FILTER_COMPARE = pd.DataFrame()
_s1b_names = [j["run_name"] for j in STAGE1B_JOBS if find_ok(j["run_name"]) is not None]
if _s1b_names:
    filt_pred_test, filt_name = get_secondary_filter()
    rows = []
    for name in _s1b_names:
        loaded = load_proba(name)
        if loaded is None:
            print("proba tidak ditemukan untuk", name)
            continue
        proba, yt, classes = loaded
        r0, r1, base, new, n_rr = apply_secondary_filter(proba, yt, classes, filt_pred_test)
        seed = int(name.rsplit("seed", 1)[1])
        for pipe, r in [("cnn_lstm_only", r0), (f"cnn_lstm_then_{filt_name}", r1)]:
            rows.append({"seed": seed, "pipeline": pipe, "accuracy": r["accuracy"], "f1_macro": r["f1_macro"],
                         "Web Attack__precision": r["Web Attack__precision"], "Web Attack__recall": r["Web Attack__recall"],
                         "Web Attack__f1": r["Web Attack__f1"], "n_rerouted": n_rr if pipe != "cnn_lstm_only" else 0})
        if seed == SEEDS[0]:
            save_confusion(yt, new, classes, FIG_DIR / f"stage1b_cm_seed42_then_{filt_name}",
                           f"CM — SMOTE no class weights + {filt_name} filter (seed 42)")
    S1B_FILTER_TABLE = pd.DataFrame(rows)
    save_table(S1B_FILTER_TABLE, "stage1b_secondary_filter_per_seed", "Tahap 1b — filter tahap kedua per seed (SMOTE no-weight)")
    agg = (S1B_FILTER_TABLE.groupby("pipeline")[["f1_macro", "Web Attack__precision", "Web Attack__recall", "Web Attack__f1"]]
           .agg(["mean", "std"]))
    agg.columns = [f"{a} {b}" for a, b in agg.columns]
    save_table(agg.reset_index(), "stage1b_secondary_filter_mean_std",
               "Tahap 1b — filter tahap kedua, mean ± std antar seed (SMOTE no-weight)")
    display(agg.reset_index())

    # Perbandingan pipeline ber-filter: CTGAN no-weight + filter vs SMOTE no-weight + filter
    filt_pipe = f"cnn_lstm_then_{filt_name}"
    if len(S1_FILTER_TABLE):
        cg = S1_FILTER_TABLE[S1_FILTER_TABLE["pipeline"] == filt_pipe]
        sg = S1B_FILTER_TABLE[S1B_FILTER_TABLE["pipeline"] == filt_pipe]
        FILTER_COMPARE = summarize([(f"CTGAN no-weight + {filt_name} (Tahap 1)", cg),
                                    (f"SMOTE no-weight + {filt_name} (Tahap 1b)", sg)])
        save_table(compact_summary(FILTER_COMPARE), "stage1b_filtered_pipelines_summary",
                   f"Pipeline ber-filter {filt_name}: CTGAN no-weight vs SMOTE no-weight (mean ± std)")
        display(compact_summary(FILTER_COMPARE))
        S1B_FILTER_TESTS = run_paired_family([(cg, sg, f"CTGAN no-weight + {filt_name}", f"SMOTE no-weight + {filt_name}")])
        if len(S1B_FILTER_TESTS):
            save_table(compact_tests(S1B_FILTER_TESTS), "stage1b_filtered_pipelines_paired_tests",
                       f"Pipeline ber-filter {filt_name}: uji berpasangan (keluarga Holm tersendiri, 4 metrik)")
            display(compact_tests(S1B_FILTER_TESTS))
    else:
        print("Tabel filter Tahap 1 belum ada — perbandingan pipeline ber-filter dilewati.")
    cm_csv = RUN_DIR / f"cm_{make_job(1, 'smote', False, SEEDS[0])['run_name']}.csv"
    if cm_csv.exists():
        shutil.copy2(cm_csv.with_suffix(".png"), guard_write(FIG_DIR / "stage1b_cm_smote_noweight_seed42.png"))
else:
    print("Filter Tahap 1b dilewati (belum ada run ok).")

Tersimpan: tables/stage1b_secondary_filter_per_seed.csv dan .md
Tersimpan: tables/stage1b_secondary_filter_mean_std.csv dan .md


,pipeline,f1_macro mean,f1_macro std,Web Attack__precision mean,Web Attack__precision std,Web Attack__recall mean,Web Attack__recall std,Web Attack__f1 mean,Web Attack__f1 std
0,cnn_lstm_only,0.8829,0.0030,0.8135,0.0293,1.0000,0.0000,0.8969,0.0179
1,cnn_lstm_then_lightgbm,0.8980,0.0006,0.9756,0.0060,1.0000,0.0000,0.9877,0.0030


Tersimpan: tables/stage1b_filtered_pipelines_summary.csv dan .md


,Model,n_seeds,Macro F1 (mean ± std),WA precision (mean ± std),WA recall (mean ± std),WA F1 (mean ± std)
0,CTGAN no-weight + lightgbm (Tahap 1),5,0.8712 ± 0.0255,0.9892 ± 0.0148,0.7444 ± 0.2366,0.8315 ± 0.1450
1,SMOTE no-weight + lightgbm (Tahap 1b),5,0.8980 ± 0.0006,0.9756 ± 0.0060,1.0000 ± 0.0000,0.9877 ± 0.0030


Tersimpan: tables/stage1b_filtered_pipelines_paired_tests.csv dan .md


,variant,reference,metric,n_pairs,variant_mean,reference_mean,mean_diff_variant_minus_ref,n_variant_better,t_pvalue,t_pvalue_holm,wilcoxon_pvalue,wilcoxon_pvalue_holm
0,CTGAN no-weight + lightgbm,SMOTE no-weight + lightgbm,Macro F1,5,0.8712,0.8980,-0.0268,2,0.0809,0.2927,0.3125,1.0000
1,CTGAN no-weight + lightgbm,SMOTE no-weight + lightgbm,WA precision,5,0.9892,0.9756,0.0136,3,0.0884,0.2927,0.2500,1.0000
2,CTGAN no-weight + lightgbm,SMOTE no-weight + lightgbm,WA recall,5,0.7444,1.0000,-0.2556,0,0.0732,0.2927,0.2500,1.0000
3,CTGAN no-weight + lightgbm,SMOTE no-weight + lightgbm,WA F1,5,0.8315,0.9877,-0.1561,0,0.0761,0.2927,0.2500,1.0000


In [19]:
# ============================================================
# TAHAP 1b — KESIMPULAN (bahasa Indonesia)
# ============================================================
STAGE1B_VERDICT = None
STAGE1B_VERDICT_LINES = []
if len(STAGE1B_TESTS) and (STAGE1B_TESTS["variant"] == KEY_VAR).any():
    key_lines, key_st = comparison_lines(STAGE1B_TESTS, KEY_VAR, KEY_REF)
    cw_lines, cw_st = comparison_lines(STAGE1B_TESTS, CW_VAR, CW_REF)
    n_c, n_s = len(S1_NOWEIGHT), len(S1B_SMOTE_NW)
    mf = key_st.get("Macro F1", "n/a")
    wf = key_st.get("WA F1", "n/a")
    if mf == "sig_lebih_tinggi":
        STAGE1B_VERDICT = ("CTGAN tanpa class weight LEBIH BAIK SECARA SIGNIFIKAN daripada SMOTE tanpa class weight "
                           "pada Macro F1 (uji t berpasangan, Holm p<0.05). Klaim 'CTGAN lebih baik dari SMOTE' "
                           "dapat dipertahankan secara adil (keduanya tanpa class weight).")
    elif mf == "rata2_lebih_tinggi":
        STAGE1B_VERDICT = ("CTGAN tanpa class weight lebih baik daripada SMOTE tanpa class weight HANYA SECARA RATA-RATA "
                           "pada Macro F1; selisihnya TIDAK signifikan setelah koreksi Holm. Di naskah, tulis sebagai "
                           "kecenderungan, bukan keunggulan yang terbukti.")
    elif mf in ("rata2_lebih_rendah", "sig_lebih_rendah", "sama"):
        STAGE1B_VERDICT = ("CTGAN tanpa class weight TIDAK lebih baik daripada SMOTE tanpa class weight pada Macro F1"
                           + (" (SMOTE justru lebih tinggi secara signifikan)" if mf == "sig_lebih_rendah" else "")
                           + ". Keunggulan CTGAN di Tahap 1 kemungkinan besar berasal dari dimatikannya class weight, bukan dari metode augmentasinya (lihat perbandingan tambahan di atas).")
    else:
        STAGE1B_VERDICT = "Perbandingan utama belum dapat diuji (pasangan seed kurang)."
    if mf in ("sig_lebih_tinggi", "rata2_lebih_tinggi") and wf == "sig_lebih_tinggi":
        STAGE1B_VERDICT += " Pada WA F1 keunggulan CTGAN juga signifikan."
    elif mf in ("sig_lebih_tinggi", "rata2_lebih_tinggi") and wf in ("rata2_lebih_rendah", "sig_lebih_rendah"):
        STAGE1B_VERDICT += " Namun pada WA F1, SMOTE lebih tinggi — perlu dibahas jujur di naskah."
    if n_c < len(SEEDS) or n_s < len(SEEDS):
        STAGE1B_VERDICT += f" PERINGATAN: seed belum lengkap (CTGAN {n_c}, SMOTE {n_s} dari {len(SEEDS)})."
    STAGE1B_VERDICT_LINES = (["PERBANDINGAN UTAMA — CTGAN tanpa class weight vs SMOTE tanpa class weight:"] + key_lines +
                             ["", "PERBANDINGAN TAMBAHAN — SMOTE tanpa class weight vs SMOTE dengan class weight "
                              "(efek class weight pada SMOTE):"] + cw_lines)
    if len(S1B_FILTER_TESTS):
        fl, _ = comparison_lines(S1B_FILTER_TESTS, S1B_FILTER_TESTS["variant"].iloc[0], S1B_FILTER_TESTS["reference"].iloc[0])
        STAGE1B_VERDICT_LINES += ["", "PIPELINE BER-FILTER (CTGAN no-weight + filter vs SMOTE no-weight + filter):"] + fl
    STAGE1B_VERDICT_LINES += ["", "KESIMPULAN TAHAP 1b: " + STAGE1B_VERDICT,
                              "(p = uji t berpasangan dengan koreksi Holm; Wilcoxon ditampilkan sebagai pelengkap. "
                              "Kesimpulan ini panduan; keputusan akhir tetap pada penulis.)"]
    print("\n".join(STAGE1B_VERDICT_LINES))
    guard_write(TABLE_DIR / "stage1b_verdict.txt").write_text("\n".join(STAGE1B_VERDICT_LINES) + "\n")
else:
    print("Kesimpulan Tahap 1b belum tersedia (butuh run Tahap 1 dan Tahap 1b berstatus ok).")

PERBANDINGAN UTAMA — CTGAN tanpa class weight vs SMOTE tanpa class weight:
  - Macro F1: 0.8655 vs 0.8829 (selisih -0.0174; lebih baik di 2/5 seed; p-t Holm=0.3045; p-Wilcoxon=0.3125) → lebih rendah rata-rata (tidak signifikan setelah Holm)
  - WA precision: 0.9164 vs 0.8135 (selisih +0.1029; lebih baik di 5/5 seed; p-t Holm=0.0735; p-Wilcoxon=0.0625) → lebih tinggi rata-rata saja (tidak signifikan setelah Holm)
  - WA recall: 0.7444 vs 1.0000 (selisih -0.2556; lebih baik di 0/5 seed; p-t Holm=0.2195; p-Wilcoxon=0.2500) → lebih rendah rata-rata (tidak signifikan setelah Holm)
  - WA F1: 0.7970 vs 0.8969 (selisih -0.1000; lebih baik di 2/5 seed; p-t Holm=0.3045; p-Wilcoxon=0.3125) → lebih rendah rata-rata (tidak signifikan setelah Holm)

PERBANDINGAN TAMBAHAN — SMOTE tanpa class weight vs SMOTE dengan class weight (efek class weight pada SMOTE):
  - Macro F1: 0.8829 vs 0.7921 (selisih +0.0907; lebih baik di 5/5 seed; p-t Holm=0.0001; p-Wilcoxon=0.0625) → LEBIH TINGGI dan SIGNIFIKAN
  - 

## Tahap 2 — CTGAN dengan kolom diskret (OPSIONAL)

> **Tahap 2 tidak perlu dijalankan sekarang.** Jalankan hanya jika diputuskan bersama setelah hasil Tahap 1b keluar. Caranya: ubah `RUN_STAGE2 = True` di sel konfigurasi paling atas, lalu **Runtime → Run all**. Run Tahap 1 yang sudah selesai akan dilewati.

Isi Tahap 2:
1. CTGAN dilatih ulang pada 290 baris Web Attack train dengan `discrete_columns` = kolom dengan ≤ 10 nilai unik (kolom konstan, port, protokol, flag), 200 epoch, batch 50, pac 10, per seed. Cache di `results_v2/synth_cache/`.
2. Mutu sintetis: KS rata-rata per fitur dan AUC diskriminator real-vs-sintetis untuk setiap seed, dibandingkan dengan CTGAN asli; TSTR (Random Forest, sama seperti sebelumnya) untuk seed 42.
3. CNN-LSTM **dengan** dan **tanpa** class weight, 5 seed masing-masing (10 run), lalu uji berpasangan vs model utama dan SMOTE dengan koreksi Holm.

In [20]:
# ============================================================
# TAHAP 2a — CTGAN kolom diskret: fit per seed + mutu sintetis + TSTR
# ============================================================
DISCRETE_COLS = select_discrete_columns(X_web_gan)
STAGE2_QUALITY = pd.DataFrame()
if RUN_STAGE2:
    print(f"Kolom diskret ({len(DISCRETE_COLS)}; kriteria <= {DISCRETE_MAX_UNIQUE} nilai unik atau "
          f"nama mengandung {DISCRETE_FORCE_KEYWORDS}):")
    print(DISCRETE_COLS)
    guard_write(RESULTS_V2_DIR / "stage2_discrete_columns.json").write_text(json.dumps(
        {"criterion": f"nunique <= {DISCRETE_MAX_UNIQUE} pada Web Attack train, atau nama mengandung {DISCRETE_FORCE_KEYWORDS}",
         "clip_to_real_range": STAGE2_CLIP_TO_REAL_RANGE, "columns": DISCRETE_COLS}, indent=2))
    q_rows = []
    q_path = TABLE_DIR / "stage2_synth_quality_per_seed.csv"
    q_done = pd.read_csv(q_path) if q_path.exists() else pd.DataFrame()
    for i, seed in enumerate(SEEDS, start=1):
        print(f"\n>>> Tahap 2 — CTGAN diskret {i} of {len(SEEDS)} (seed {seed})")
        Xs_d, meta_d = get_synth_discrete(seed)
        for kind, Xs in [("ctgan_discrete", Xs_d), ("ctgan_original", None)]:
            if len(q_done) and ((q_done["tag"] == kind) & (q_done["seed"] == seed)).any():
                q_rows.append(q_done[(q_done["tag"] == kind) & (q_done["seed"] == seed)].iloc[0].to_dict())
                continue
            if Xs is None:
                try:
                    Xs, _ = get_synth_original(seed)
                except Exception as exc:
                    print("Sintetis CTGAN asli tidak tersedia untuk pembanding:", exc)
                    continue
            rec, ks_df = evaluate_synth_quality(X_web_gan, Xs, tag=kind, seed=seed)
            ks_df.to_csv(guard_write(SYNTH_DIR / f"ks_per_feature_{kind}_seed{seed}.csv"), index=False)
            q_rows.append(rec)
        pd.DataFrame(q_rows).to_csv(guard_write(q_path), index=False)
    STAGE2_QUALITY = pd.DataFrame(q_rows)
    qagg = STAGE2_QUALITY.groupby("tag")[["ks_mean", "ks_median", "n_features_ks_gt_0_5", "discriminator_auc",
                                         "corr_absdiff_mean", "pct_values_outside_0_1"]].agg(["mean", "std"])
    qagg.columns = [f"{a} {b}" for a, b in qagg.columns]
    qagg = qagg.reset_index()
    save_table(STAGE2_QUALITY, "stage2_synth_quality_per_seed", "Tahap 2 — mutu sintetis per seed")
    save_table(qagg, "stage2_synth_quality_mean_std", "Tahap 2 — mutu sintetis, mean ± std antar seed")
    display(qagg)

    # TSTR seed pertama (mahal: RF pada ~2,5 juta baris). Hasil CTGAN asli diambil dari results/tstr_rf_web.csv.
    tstr_path = TABLE_DIR / "stage2_tstr.csv"
    if RUN_TSTR_STAGE2 and not tstr_path.exists():
        Xs_d, _ = get_synth_discrete(SEEDS[0])
        rows_t = [run_tstr(Xs_d, tag=f"ctgan_discrete_seed{SEEDS[0]}")]
        old_tstr = OLD_RESULTS_DIR / "tstr_rf_web.csv"
        if old_tstr.exists():
            o = pd.read_csv(old_tstr).iloc[0].to_dict()
            o["tag"] = f"ctgan_original_seed{SEEDS[0]} (results/tstr_rf_web.csv)"
            rows_t.append(o)
        keep = ["tag", "accuracy", "f1_macro", "Web Attack__precision", "Web Attack__recall", "Web Attack__f1", "fit_seconds"]
        tstr_df = pd.DataFrame(rows_t)
        save_table(tstr_df[[c for c in keep if c in tstr_df.columns]], "stage2_tstr",
                   "Tahap 2 — TSTR (RF: non-Web train asli + Web sintetis, uji pada test asli)")
    if tstr_path.exists():
        display(pd.read_csv(tstr_path))
else:
    print("Tahap 2 tidak dijalankan (RUN_STAGE2=False).")

Tahap 2 tidak dijalankan (RUN_STAGE2=False).


In [21]:
# ============================================================
# TAHAP 2b — CNN-LSTM dengan & tanpa class weight pada sintetis diskret (10 run)
# ============================================================
STAGE2_JOBS = []
for s in SEEDS:
    STAGE2_JOBS.append(make_job(2, "discrete", True, s))
    STAGE2_JOBS.append(make_job(2, "discrete", False, s))
if RUN_STAGE2:
    run_jobs(STAGE2_JOBS, "Tahap 2")
else:
    print("Tahap 2 dilewati (RUN_STAGE2=False).")

Tahap 2 dilewati (RUN_STAGE2=False).


In [22]:
# ============================================================
# TAHAP 2c — ringkasan dan uji berpasangan (Holm atas seluruh keluarga Tahap 2)
# ============================================================
STAGE2_SUMMARY = pd.DataFrame()
STAGE2_TESTS = pd.DataFrame()
S2_W1 = v2_group("discrete", True, 2)
S2_W0 = v2_group("discrete", False, 2)
if RUN_STAGE2 and (len(S2_W1) or len(S2_W0)):
    groups = [("CTGAN-discrete + class weights (v2, Tahap 2)", S2_W1),
              ("CTGAN-discrete, no class weights (v2, Tahap 2)", S2_W0)]
    if len(S1_NOWEIGHT):
        groups.append((S1_LABEL, S1_NOWEIGHT))
    for key in ["main_ctgan", "smote"]:
        if key in OLD_REFS:
            groups.append((OLD_REFS[key]["model_label"].iloc[0], OLD_REFS[key]))
    STAGE2_SUMMARY = summarize(groups)
    save_table(STAGE2_SUMMARY, "stage2_summary_full", "Tahap 2 — mean ± std (lengkap)")
    save_table(compact_summary(STAGE2_SUMMARY), "stage2_summary", "Tahap 2 — mean ± std (5 seed)")
    display(compact_summary(STAGE2_SUMMARY))
    pairs = []
    for vdf, vl in [(S2_W1, "CTGAN-discrete + weights"), (S2_W0, "CTGAN-discrete no-weight")]:
        if "main_ctgan" in OLD_REFS:
            pairs.append((vdf, OLD_REFS["main_ctgan"], vl, "CTGAN + weights (main)"))
        if "smote" in OLD_REFS:
            pairs.append((vdf, OLD_REFS["smote"], vl, "SMOTE + weights"))
    STAGE2_TESTS = run_paired_family(pairs)
    if len(STAGE2_TESTS):
        save_table(STAGE2_TESTS, "stage2_paired_tests_full", "Tahap 2 — uji berpasangan (lengkap)")
        save_table(compact_tests(STAGE2_TESTS), "stage2_paired_tests",
                   "Tahap 2 — uji t berpasangan & Wilcoxon, koreksi Holm atas seluruh keluarga uji")
        display(compact_tests(STAGE2_TESTS))
else:
    print("Ringkasan Tahap 2 dilewati.")

Ringkasan Tahap 2 dilewati.


In [23]:
# ============================================================
# RINGKASAN AKHIR + saran lanjut + bukti folder results/ tidak berubah
# ============================================================
def _fmt(df, label, metric):
    if not len(df):
        return "—"
    r = df[df["Model"] == label]
    if not len(r):
        return "—"
    return r.iloc[0].get(f"{METRIC_LABEL[metric]} (mean ± std)", "—")


print("=" * 72)
print("RINGKASAN EKSPERIMEN v3 (keluaran di results_v2)" + ("  [SMOKE — JANGAN DIPAKAI]" if SMOKE else ""))
print("=" * 72)
runs_df = pd.DataFrame(ALL_ROWS)
if len(runs_df):
    runs_df = runs_df.drop_duplicates("run_name", keep="last")
    print(f"Run di all_runs_v2.csv: {len(runs_df)} | ok={int(runs_df['status'].eq('ok').sum())} | "
          f"error={int(runs_df['status'].eq('error').sum())}")
    cols = [c for c in ["run_name", "status", "f1_macro", "Web Attack__precision", "Web Attack__recall",
                        "Web Attack__f1", "n_epochs_ran", "train_seconds"] if c in runs_df.columns]
    display(runs_df[cols])

verdict = None
if len(STAGE1_SUMMARY):
    print("\nTahap 1 (mean ± std):")
    display(compact_summary(STAGE1_SUMMARY))
    if len(STAGE1_TESTS):
        display(compact_tests(STAGE1_TESTS))
    if STAGE1_VERDICT_LINES:
        print("\nKesimpulan Tahap 1 per perbandingan:")
        print("\n".join(STAGE1_VERDICT_LINES))
        verdict = " | ".join(l.strip() for l in STAGE1_VERDICT_LINES)
if len(S1_FILTER_TABLE):
    print("\nFilter tahap kedua (Tahap 1, seed 42):")
    display(S1_FILTER_TABLE[S1_FILTER_TABLE["seed"] == SEEDS[0]])
if len(STAGE1B_SUMMARY):
    print("\nTahap 1b (mean ± std):")
    display(compact_summary(STAGE1B_SUMMARY))
    if len(STAGE1B_TESTS):
        display(compact_tests(STAGE1B_TESTS))
    if len(FILTER_COMPARE):
        print("\nPipeline ber-filter (mean ± std):")
        display(compact_summary(FILTER_COMPARE))
    if STAGE1B_VERDICT_LINES:
        print("\n" + "\n".join(STAGE1B_VERDICT_LINES))
if len(STAGE2_SUMMARY):
    print("\nTahap 2 (mean ± std):")
    display(compact_summary(STAGE2_SUMMARY))
    if len(STAGE2_TESTS):
        display(compact_tests(STAGE2_TESTS))

# Bukti: tidak ada berkas di results/ yang ditambah/diubah/dihapus oleh notebook ini
now_snap = snapshot_old_results()
changed = sorted(k for k in set(OLD_RESULTS_SNAPSHOT) | set(now_snap) if OLD_RESULTS_SNAPSHOT.get(k) != now_snap.get(k))
print("\nFolder results/ lama tidak berubah:" , "YA" if not changed else f"TIDAK — {changed[:10]}")

manifest = {
    "smoke": SMOKE, "finished_iso": datetime.now().isoformat(timespec="seconds"),
    "notebook_version": NOTEBOOK_VERSION, "gpu_name_this_session": GPU_NAME,
    "gpu_previous_sessions": [h.get("gpu_name") for h in ENV_INFO.get("previous_sessions", [])],
    "run_stage1": RUN_STAGE1, "run_stage1b": RUN_STAGE1B, "run_stage2": RUN_STAGE2, "seeds": SEEDS, "n_synth": N_SYNTH,
    "dl_epochs": DL_EPOCHS, "dl_patience": DL_PATIENCE, "dl_batch_size": DL_BATCH_SIZE,
    "stage1_verdict": verdict, "stage1b_verdict": STAGE1B_VERDICT, "old_results_unchanged": not changed,
    "files": sorted(str(p.relative_to(RESULTS_V2_DIR)) for p in RESULTS_V2_DIR.rglob("*") if p.is_file()),
}
guard_write(RESULTS_V2_DIR / "run_manifest_v2.json").write_text(json.dumps(manifest, indent=2))
print("\nSemua keluaran ada di:", RESULTS_V2_DIR)
print("Kirim folder tables/ dan figures/, serta all_runs_v2.csv dan environment.json (file kecil). "
      "File proba_*.npz dan synth_cache/ tidak perlu dikirim.")
if GPU_NAME and "A100" not in str(GPU_NAME).upper():
    print(f"\nPENGINGAT: sesi ini memakai GPU '{GPU_NAME}', bukan A100. Sebutkan di bagian perangkat keras naskah.")
# GPU per run (run lama dari v2 tidak punya kolom gpu_name; lihat environment.json -> previous_sessions)
if len(runs_df) and "gpu_name" in runs_df.columns:
    print("\nGPU per run:")
    display(runs_df[["run_name", "gpu_name"]].fillna({"gpu_name": "(v2 — lihat environment.json previous_sessions)"}))
if SMOKE:
    print("\n*** PERINGATAN: mode smoke / data palsu. Jangan masukkan angka ini ke makalah. ***")

RINGKASAN EKSPERIMEN v3 (keluaran di results_v2)
Run di all_runs_v2.csv: 10 | ok=10 | error=0


,run_name,status,f1_macro,Web Attack__precision,Web Attack__recall,Web Attack__f1,n_epochs_ran,train_seconds
0,s1_main_ctgan_cnn_lstm_w0_n5000_seed42,ok,0.8591,0.9375,0.6250,0.7500,50,1423.7709
1,s1_main_ctgan_cnn_lstm_w0_n5000_seed123,ok,0.8407,0.9737,0.5139,0.6727,29,837.1384
2,s1_main_ctgan_cnn_lstm_w0_n5000_seed456,ok,0.8920,0.9114,1.0000,0.9536,38,1105.6394
3,s1_main_ctgan_cnn_lstm_w0_n5000_seed789,ok,0.8553,0.9767,0.5833,0.7304,44,1271.4740
4,s1_main_ctgan_cnn_lstm_w0_n5000_seed1024,ok,0.8803,0.7826,1.0000,0.8780,39,1133.6440
5,s1b_smote_cnn_lstm_w0_n5000_seed42,ok,0.8868,0.8471,1.0000,0.9172,30,755.1633
6,s1b_smote_cnn_lstm_w0_n5000_seed123,ok,0.8822,0.8090,1.0000,0.8944,33,806.1368
7,s1b_smote_cnn_lstm_w0_n5000_seed456,ok,0.8810,0.8000,1.0000,0.8889,28,678.8789
8,s1b_smote_cnn_lstm_w0_n5000_seed789,ok,0.8849,0.8372,1.0000,0.9114,30,729.4903
9,s1b_smote_cnn_lstm_w0_n5000_seed1024,ok,0.8793,0.7742,1.0000,0.8727,32,785.4861



Tahap 1 (mean ± std):


,Model,n_seeds,Macro F1 (mean ± std),WA precision (mean ± std),WA recall (mean ± std),WA F1 (mean ± std)
0,"Targeted CTGAN, no class weights (v2, Tahap 1)",5,0.8655 ± 0.0205,0.9164 ± 0.0795,0.7444 ± 0.2366,0.7970 ± 0.1153
1,"Targeted CTGAN + class weights (Tabel 16, resu...",5,0.8176 ± 0.0334,0.3973 ± 0.1900,1.0000 ± 0.0000,0.5465 ± 0.2050
2,SMOTE + class weights (results/),5,0.7921 ± 0.0082,0.2450 ± 0.0684,1.0000 ± 0.0000,0.3899 ± 0.0849
3,"No augmentation, class weights only (results/)",5,0.7288 ± 0.0476,0.1282 ± 0.2680,0.8944 ± 0.2360,0.1195 ± 0.2312
4,Undersample only (results/),5,0.7239 ± 0.0422,0.0886 ± 0.0971,0.8917 ± 0.2346,0.1429 ± 0.1527
5,"Tanpa class weight, 1 seed (Tabel 15, results/)",1,0.8829,0.8161,0.9861,0.8931


,variant,reference,metric,n_pairs,variant_mean,reference_mean,mean_diff_variant_minus_ref,n_variant_better,t_pvalue,t_pvalue_holm,wilcoxon_pvalue,wilcoxon_pvalue_holm
0,CTGAN no-weight (v2),CTGAN + weights (main),Macro F1,5,0.8655,0.8176,0.0479,5,0.0444,0.1776,0.0625,0.5000
1,CTGAN no-weight (v2),CTGAN + weights (main),WA precision,5,0.9164,0.3973,0.5191,5,0.0062,0.0310,0.0625,0.5000
2,CTGAN no-weight (v2),CTGAN + weights (main),WA recall,5,0.7444,1.0000,-0.2556,0,0.0732,0.2195,0.2500,0.5000
3,CTGAN no-weight (v2),CTGAN + weights (main),WA F1,5,0.7970,0.5465,0.2505,4,0.0749,0.2195,0.1250,0.5000
4,CTGAN no-weight (v2),SMOTE + weights,Macro F1,5,0.8655,0.7921,0.0733,5,0.0022,0.0151,0.0625,0.5000
5,CTGAN no-weight (v2),SMOTE + weights,WA precision,5,0.9164,0.2450,0.6714,5,0.0001,0.0006,0.0625,0.5000
6,CTGAN no-weight (v2),SMOTE + weights,WA recall,5,0.7444,1.0000,-0.2556,0,0.0732,0.2195,0.2500,0.5000
7,CTGAN no-weight (v2),SMOTE + weights,WA F1,5,0.7970,0.3899,0.4071,5,0.0041,0.0243,0.0625,0.5000



Kesimpulan Tahap 1 per perbandingan:
CTGAN tanpa class weight vs CTGAN + weights (main):
  - Macro F1: 0.8655 vs 0.8176 (selisih +0.0479; lebih baik di 5/5 seed; p-t Holm=0.1776; p-Wilcoxon=0.0625) → lebih tinggi rata-rata saja (tidak signifikan setelah Holm)
  - WA precision: 0.9164 vs 0.3973 (selisih +0.5191; lebih baik di 5/5 seed; p-t Holm=0.0310; p-Wilcoxon=0.0625) → LEBIH TINGGI dan SIGNIFIKAN
  - WA recall: 0.7444 vs 1.0000 (selisih -0.2556; lebih baik di 0/5 seed; p-t Holm=0.2195; p-Wilcoxon=0.2500) → lebih rendah rata-rata (tidak signifikan setelah Holm)
  - WA F1: 0.7970 vs 0.5465 (selisih +0.2505; lebih baik di 4/5 seed; p-t Holm=0.2195; p-Wilcoxon=0.1250) → lebih tinggi rata-rata saja (tidak signifikan setelah Holm)
CTGAN tanpa class weight vs SMOTE + weights:
  - Macro F1: 0.8655 vs 0.7921 (selisih +0.0733; lebih baik di 5/5 seed; p-t Holm=0.0151; p-Wilcoxon=0.0625) → LEBIH TINGGI dan SIGNIFIKAN
  - WA precision: 0.9164 vs 0.2450 (selisih +0.6714; lebih baik di 5/5 seed; 

... (4 outputs left out)


Pipeline ber-filter (mean ± std):


,Model,n_seeds,Macro F1 (mean ± std),WA precision (mean ± std),WA recall (mean ± std),WA F1 (mean ± std)
0,CTGAN no-weight + lightgbm (Tahap 1),5,0.8712 ± 0.0255,0.9892 ± 0.0148,0.7444 ± 0.2366,0.8315 ± 0.1450
1,SMOTE no-weight + lightgbm (Tahap 1b),5,0.8980 ± 0.0006,0.9756 ± 0.0060,1.0000 ± 0.0000,0.9877 ± 0.0030



PERBANDINGAN UTAMA — CTGAN tanpa class weight vs SMOTE tanpa class weight:
  - Macro F1: 0.8655 vs 0.8829 (selisih -0.0174; lebih baik di 2/5 seed; p-t Holm=0.3045; p-Wilcoxon=0.3125) → lebih rendah rata-rata (tidak signifikan setelah Holm)
  - WA precision: 0.9164 vs 0.8135 (selisih +0.1029; lebih baik di 5/5 seed; p-t Holm=0.0735; p-Wilcoxon=0.0625) → lebih tinggi rata-rata saja (tidak signifikan setelah Holm)
  - WA recall: 0.7444 vs 1.0000 (selisih -0.2556; lebih baik di 0/5 seed; p-t Holm=0.2195; p-Wilcoxon=0.2500) → lebih rendah rata-rata (tidak signifikan setelah Holm)
  - WA F1: 0.7970 vs 0.8969 (selisih -0.1000; lebih baik di 2/5 seed; p-t Holm=0.3045; p-Wilcoxon=0.3125) → lebih rendah rata-rata (tidak signifikan setelah Holm)

PERBANDINGAN TAMBAHAN — SMOTE tanpa class weight vs SMOTE dengan class weight (efek class weight pada SMOTE):
  - Macro F1: 0.8829 vs 0.7921 (selisih +0.0907; lebih baik di 5/5 seed; p-t Holm=0.0001; p-Wilcoxon=0.0625) → LEBIH TINGGI dan SIGNIFIKAN
  -

,run_name,gpu_name
0,s1_main_ctgan_cnn_lstm_w0_n5000_seed42,(v2 — lihat environment.json previous_sessions)
1,s1_main_ctgan_cnn_lstm_w0_n5000_seed123,(v2 — lihat environment.json previous_sessions)
2,s1_main_ctgan_cnn_lstm_w0_n5000_seed456,(v2 — lihat environment.json previous_sessions)
3,s1_main_ctgan_cnn_lstm_w0_n5000_seed789,(v2 — lihat environment.json previous_sessions)
4,s1_main_ctgan_cnn_lstm_w0_n5000_seed1024,(v2 — lihat environment.json previous_sessions)
5,s1b_smote_cnn_lstm_w0_n5000_seed42,NVIDIA A100-SXM4-80GB
6,s1b_smote_cnn_lstm_w0_n5000_seed123,NVIDIA A100-SXM4-80GB
7,s1b_smote_cnn_lstm_w0_n5000_seed456,NVIDIA A100-SXM4-80GB
8,s1b_smote_cnn_lstm_w0_n5000_seed789,NVIDIA A100-SXM4-80GB
9,s1b_smote_cnn_lstm_w0_n5000_seed1024,NVIDIA A100-SXM4-80GB


## Catatan

- Test set selalu dari split 80/20 `random_state=42` dan tidak diubah; sama dengan `results/`.
- Folder `results/` hanya dibaca. Sel ringkasan akhir memeriksa bahwa tidak ada berkas di sana yang berubah.
- Tahap 1b memakai data latih yang identik dengan baseline SMOTE di `results/` (SMOTE k=5, `random_state=seed`, Web Attack 290 → 5290, undersample Benign seed yang sama); hanya class weight yang dimatikan.
- Semua uji berpasangan memakai seed yang sama (42, 123, 456, 789, 1024). Koreksi Holm dihitung per tahap atas semua perbandingan × metrik (Macro F1, WA precision, WA recall, WA F1), terpisah untuk uji t dan Wilcoxon.
- Dengan 5 seed, p Wilcoxon dua sisi terkecil yang mungkin adalah 0.0625; ini batas uji, bukan kelemahan hasil.
- Jika sesi terputus, jalankan semua sel lagi dari atas; run yang sudah `ok` tidak diulang, cache CTGAN dan `proba_*.npz` dibaca dari `results_v2/`.